# Bateria de cenários — validação em dados simulados

Conjunto sistemático de cenários para caracterizar o modelo em todo o espaço de
comportamentos, com verdade conhecida (parâmetros geradores fixos). Cada
cenário corre a avaliação completa (`avaliar_serie_simulada`): série criada,
recuperação (conhecido/desconhecido) com gráficos de recriação, uma
identificabilidade ilustrativa numa réplica de referência, e um estudo Monte
Carlo emparelhado (viés/DP/rmse + cobertura + identificabilidade agregada
sobre várias réplicas).

**Este notebook é autocontido**: as funções nucleares do modelo estão
definidas nas suas próprias células, sem depender de nenhum módulo externo. A
secção "especificidades dos dados simulados" acrescenta a geração de séries com
verdade conhecida, a comparação de métodos de recuperação e a identificabilidade
agregada sobre `K_identif` réplicas por cenário.

## 1. Núcleo do modelo

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
from scipy.ndimage import label
from scipy.optimize import minimize
from scipy.stats import chi2
from sklearn.linear_model import LinearRegression


DIAS_UTEIS_ANO = 252

def simular_processo(T, dt, x0, params_regime1, params_regime2, theta, regime_inicial=1):
    '''Simula uma trajetória do modelo com mudança de regime por limiares afins.

    Em cada passo, o regime é atualizado com base no preço e nos limiares do 
    instante anterior(versão discreta da regra de comutação, adequada para dt
    pequeno) e o preço evolui pela solução exata do MBG nesse regime.
    O ponto que dispara a mudança (x[i-1], já para lá da fronteira)
    foi gerado com os parâmetros do regime cessante. É o ponto seguinte, x[i],
    que é rotulado em regimes[i] e gerado já com os parâmetros do regime novo.

    Parâmetros
    ----------
    T : float
        Horizonte da simulação, em anos.
    dt : float
        Passo temporal
    x0 : float
        Preço inicial.
    params_regime1, params_regime2 : tuple (mu, sigma)
        Drift e volatilidade anualizados de cada regime.
    theta : sequência (a1, b1, a2, b2)
        Declives e ordenadas na origem de M(t) e m(t).
    regime_inicial : int
        Regime em t = 0.

    Devolve
    -------
    (t, x, regimes) : arrays com os instantes, os preços e o regime em vigor
    em cada instante.
    '''
    n = int(T / dt)
    t = np.arange(n) * dt
    x = np.zeros(n)
    regimes = np.zeros(n, dtype=int)
    x[0] = x0
    regime = regime_inicial
    regimes[0] = regime
    parametros = {1: params_regime1, 2: params_regime2}
    a1, b1, a2, b2 = theta
    for i in range(1, n):
        L_M = a1 * t[i-1] + b1
        L_m = a2 * t[i-1] + b2
        if regime == 1 and x[i-1] >= L_M:
            regime = 2
        elif regime == 2 and x[i-1] <= L_m:
            regime = 1
        regimes[i] = regime
        mu, sigma = parametros[regime]
        dB = np.random.normal(0, np.sqrt(dt))
        x[i] = x[i-1] * np.exp((mu - 0.5 * sigma**2) * dt + sigma * dB)
    return t, x, regimes

def _classificar_regimes(t, x, theta, regime_inicial=1):
    '''Atribui a cada instante o regime em vigor, dados os limiares theta.

    Aplica a mesma regra de comutação do simulador: no regime 1 passa-se ao
    regime 2 quando x[i-1] >= M(t[i-1]); no regime 2 passa-se ao regime 1
    quando x[i-1] <= m(t[i-1]).

    Parâmetros
    ----------
    t, x : array
        Instantes (anos) e preços observados.
    theta : sequência (a1, b1, a2, b2)
        Declives e ordenadas na origem de M(t) e m(t).
    regime_inicial : int
        Regime assumido em t[0].

    Devolve
    -------
    regimes : array de inteiros em {1, 2}, um por instante de t.
    '''
    a1, b1, a2, b2 = theta
    n = len(x)
    regime = regime_inicial
    regimes = np.zeros(n, dtype=int)
    regimes[0] = regime
    for i in range(1, n):
        L_M = a1 * t[i-1] + b1
        L_m = a2 * t[i-1] + b2
        if regime == 1 and x[i-1] >= L_M:
            regime = 2
        elif regime == 2 and x[i-1] <= L_m:
            regime = 1
        regimes[i] = regime
    return regimes


def _mle_por_regime(x, regimes, dt):
    '''Estimadores de máxima verosimilhança de (mu_r, sigma_r), condicionais a
    uma classificação em regimes fixa.

    O log-retorno log(x[k+1]/x[k]) é atribuído a regimes[k+1], o regime
    vigente no início desse intervalo. Os estimadores são
    anualizados pela divisão por dt, e a log-verosimilhança é a da densidade
    lognormal de X.

    Parâmetros
    ----------
    x : array
        Preços observados.
    regimes : array de inteiros em {1, 2}
        Regime em vigor em cada instante (mesmo comprimento de x).
    dt : float
        Passo temporal

    Devolve
    -------
    (resultados, log_lik_total):
        resultados : dict {1: {...}, 2: {...}}
            Por regime r, um dicionário com:
                'mu' : float ou None
                    Estimativa anualizada de mu_r; None se n_r < 2.
                'sigma' : float ou None
                    Estimativa anualizada de sigma_r; None se n_r < 2.
                'n_obs' : int
                    Número de log-retornos atribuídos ao regime r (n_r).
                'log_lik' : float
                    Contribuição do regime r para a log-verosimilhança
                    total; 0 se n_r < 2.
        log_lik_total : float
            Soma das contribuições 'log_lik' dos dois regimes.
    '''
    EPSILON = 1e-9
    log_retornos = np.diff(np.log(x))
    resultados = {}
    log_lik_total = 0.0
    for r in [1, 2]:
        no_regime = (regimes[1:] == r)
        z = log_retornos[no_regime]
        x_seguinte = x[1:][no_regime]
        n_r = len(z)
        if n_r > 1:
            sigma2 = max(np.var(z) / dt, EPSILON)        
            sigma = np.sqrt(sigma2)
            mu = np.mean(z) / dt + sigma2 / 2           
            residuos = z - (mu - 0.5 * sigma2) * dt
            log_lik = (- np.sum(np.log(x_seguinte))
                       + n_r * (-0.5 * np.log(2 * np.pi * sigma2 * dt))
                       - np.sum(residuos**2) / (2 * sigma2 * dt))
            log_lik_total += log_lik
            resultados[r] = {'mu': mu, 'sigma': sigma, 'n_obs': n_r, 'log_lik': log_lik}
        else:
            resultados[r] = {'mu': None, 'sigma': None, 'n_obs': n_r, 'log_lik': 0}
    return resultados, log_lik_total


def classificar_e_estimar(t, x, theta):
    '''Classifica a série em regimes com os limiares theta e estima (mu_r, sigma_r).

    Como o regime em t = 0 não é observável, testam-se ambos os regimes
    iniciais e retém-se a classificação com maior log-verosimilhança (a escolha
    só é relevante quando x[0] está entre os limiares).

    Parâmetros
    ----------
    t, x : array
        Instantes (anos) e preços observados.
    theta : sequência (a1, b1, a2, b2)
        Declives e ordenadas na origem de M(t) e m(t).

    Devolve
    -------
    (melhor_res, melhor_regimes, melhor_lik):
        melhor_res : dict {1: {...}, 2: {...}}
            As estimativas de mu, sigma, n_obs e log_lik de AMBOS os
            regimes, provenientes da classificação (regime_inicial=1 ou
            2) que obteve a maior log-verosimilhança total.
        melhor_regimes : array de inteiros em {1, 2}
            A classificação completa correspondente a essa mesma
            corrida vencedora, uma etiqueta por observação de x.
        melhor_lik : float
            A log-verosimilhança total dessa classificação (soma dos
            dois regimes).
    '''
    dt = t[1] - t[0]
    melhor_lik, melhor_res, melhor_regimes = -np.inf, None, None
    for regime_inicial in [1, 2]:
        regimes = _classificar_regimes(t, x, theta, regime_inicial)
        resultados, lik = _mle_por_regime(x, regimes, dt)
        if lik > melhor_lik:
            melhor_lik, melhor_res, melhor_regimes = lik, resultados, regimes
    return melhor_res, melhor_regimes, melhor_lik

def _ajustar_retas(t_resist, x_resist, t_suporte, x_suporte, metodo, hiperparametros, T):
    '''Ajusta por mínimos quadrados a reta M(t) aos pontos (t_resist, x_resist)
    e a reta m(t) aos pontos (t_suporte, x_suporte).

    A semente só é aceite se M(t) > m(t) em todo o intervalo [0, T].

    Parâmetros
    ----------
    t_resist, x_resist : array
        Pontos de controlo de M(t).
    t_suporte, x_suporte : array
        Pontos de controlo de m(t).
    metodo : str
        Nome da estratégia de geração de sementes que produziu estes pontos
        (guardado na semente, apenas informativo).
    hiperparametros : dict
        Hiperparâmetros da estratégia (guardados na semente, apenas informativo).
    T : float
        Horizonte da série, em anos.

    Devolve
    -------
    dict ou None:
        Se a semente for aceite, um dicionário com:
            'metodo' : str
                O valor recebido em `metodo`.
            chaves de `hiperparametros`
                Os hiperparâmetros dessa estratégia, copiados tal como
                recebidos.
            'a1', 'b1' : float
                Declive e ordenada na origem da reta ajustada a
                (t_resist, x_resist) — M(t) = a1*t + b1.
            'a2', 'b2' : float
                Declive e ordenada na origem da reta ajustada a
                (t_suporte, x_suporte) — m(t) = a2*t + b2.
        None, se houver menos de dois pontos numa das retas, ou se a
        largura da banda ajustada não for positiva em todo o [0, T].
    '''
    if len(t_resist) < 2 or len(t_suporte) < 2:
        return None
    reta_resist = LinearRegression().fit(np.array(t_resist).reshape(-1, 1), x_resist)
    reta_suporte = LinearRegression().fit(np.array(t_suporte).reshape(-1, 1), x_suporte)
    theta = [reta_resist.coef_[0], reta_resist.intercept_,
             reta_suporte.coef_[0], reta_suporte.intercept_]
    if _largura_minima(theta, T) > 0:
        semente = {'metodo': metodo}
        semente.update(hiperparametros)
        semente.update({'a1': theta[0], 'b1': theta[1], 'a2': theta[2], 'b2': theta[3]})
        return semente
    return None


def _pontos_desfasados(t, x, indices, delta):
    '''Coordenadas (t, x) dos pontos em índices + delta, truncados ao domínio da série.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série completa.
    indices : sequência de int
        Índices de referência (tipicamente extremos locais).
    delta : int
        Deslocamento temporal aplicado a cada índice, em observações.

    Devolve
    -------
    (t_desfasado, x_desfasado):
        t_desfasado : list de float
            t[i], onde i = índice de referência + delta, forçado a ficar
            dentro de [0, len(x)-1] — evita sair dos limites do array
            quando o índice deslocado cai fora da série.
        x_desfasado : list de float
            x[i], no mesmo índice i (já corrigido) de cada ponto.
    '''
    n = len(x)
    idx = [max(0, min(n - 1, p + delta)) for p in indices]
    return [t[i] for i in idx], [x[i] for i in idx]


def sementes_janela_fixa(t, x, w_list, alpha_list, delta_list):
    '''Estratégia A — extremos locais em janelas fixas.

    A série é dividida em janelas consecutivas de w observações. Em cada janela,
    são candidatos a M(t) os pontos com x >= (1 - alpha) * máximo e ao
    m(t) os pontos com x <= (1 + alpha) * mínimo. Os pontos são
    depois desfasados delta observações. Cada terno (w, alpha, delta)
    produz, no máximo, uma semente.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    w_list : sequência de int
        Comprimentos de janela a testar.
    alpha_list : sequência de float
        Margens de tolerância em torno do extremo de cada janela.
    delta_list : sequência de int
        Deslocamentos temporais a aplicar aos extremos.

    Devolve
    -------
    DataFrame, com uma linha por semente admissível e as colunas:
        'metodo' : str
            Sempre 'janela_fixa'.
        'w', 'alpha', 'delta' : int, float, int
            Os hiperparâmetros que geraram essa semente.
        'a1', 'b1', 'a2', 'b2' : float
            Declive e ordenada na origem de M(t) e de m(t) ajustadas.
        Pode ter menos linhas do que len(w_list) × len(alpha_list) ×
        len(delta_list): combinações sem pontos suficientes numa janela, ou
        com banda inadmissível.
    '''
    T = t[-1]
    sementes = []
    for w in w_list:
        for alpha in alpha_list:
            idx_max, idx_min = [], []
            for inicio in range(0, len(x), w):
                janela = x[inicio:inicio + w]
                if len(janela) < 2:
                    continue
                idx_max += [inicio + j for j in np.where(janela >= (1 - alpha) * np.max(janela))[0]]
                idx_min += [inicio + j for j in np.where(janela <= (1 + alpha) * np.min(janela))[0]]
            if not idx_max or not idx_min:
                continue
            for delta in delta_list:
                t_res, x_res = _pontos_desfasados(t, x, idx_max, delta)
                t_sup, x_sup = _pontos_desfasados(t, x, idx_min, delta)
                s = _ajustar_retas(t_res, x_res, t_sup, x_sup, 'janela_fixa',
                                   {'w': w, 'alpha': alpha, 'delta': delta}, T)
                if s:
                    sementes.append(s)
    return pd.DataFrame(sementes)


def sementes_janela_monotonica(t, x, w_list, alpha_list, delta_list):
    '''Estratégia B — como a estratégia A, mas os extremos de uma janela só são
    aproveitados se o seu máximo (mínimo) exceder (for inferior a) o da janela
    anterior, o que aproxima uma envolvente monotónica da série.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    w_list : sequência de int
        Comprimentos de janela a testar.
    alpha_list : sequência de float
        Margens de tolerância em torno do extremo de cada janela.
    delta_list : sequência de int
        Deslocamentos temporais a aplicar aos extremos.

    Devolve
    -------
    DataFrame, com uma linha por semente admissível e as colunas:
        'metodo' : str
            Sempre 'janela_monotonica'.
        'w', 'alpha', 'delta' : int, float, int
            Os hiperparâmetros que geraram essa semente.
        'a1', 'b1', 'a2', 'b2' : float
            Declive e ordenada na origem de M(t) e de m(t) ajustadas.
        Mesmo formato de sementes_janela_fixa, pode ter menos linhas do
        que len(w_list) × len(alpha_list) × len(delta_list).
    '''
    T = t[-1]
    sementes = []
    for w in w_list:
        for alpha in alpha_list:
            idx_max, idx_min = [], []
            max_anterior, min_anterior = -np.inf, np.inf
            for inicio in range(0, len(x), w):
                janela = x[inicio:inicio + w]
                if len(janela) < 2:
                    continue
                max_local, min_local = np.max(janela), np.min(janela)
                if max_local > max_anterior:
                    idx_max += [inicio + j for j in np.where(janela >= (1 - alpha) * max_local)[0]]
                if min_local < min_anterior:
                    idx_min += [inicio + j for j in np.where(janela <= (1 + alpha) * min_local)[0]]
                max_anterior, min_anterior = max_local, min_local
            if not idx_max or not idx_min:
                continue
            for delta in delta_list:
                t_res, x_res = _pontos_desfasados(t, x, idx_max, delta)
                t_sup, x_sup = _pontos_desfasados(t, x, idx_min, delta)
                s = _ajustar_retas(t_res, x_res, t_sup, x_sup, 'janela_monotonica',
                                   {'w': w, 'alpha': alpha, 'delta': delta}, T)
                if s:
                    sementes.append(s)
    return pd.DataFrame(sementes)


def sementes_percentil(t, x, w_list, p_superior=99, p_inferior=1):
    '''Estratégia C — percentis por janela.

    Em cada janela de w observações, são candidatos a M(t) os pontos
    acima do percentil p_superior e a m(t) os pontos abaixo do percentil
    p_inferior. Cada w produz, no máximo, uma semente.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    w_list : sequência de int
        Comprimentos de janela a testar.
    p_superior, p_inferior : float
        Percentis (0-100) que definem os pontos candidatos a M(t) e a
        m(t), respetivamente.

    Devolve
    -------
    DataFrame, com uma linha por semente admissível e as colunas:
        'metodo' : str
            Sempre 'percentil'.
        'w' : int
            O comprimento de janela que gerou essa semente.
        'p_superior', 'p_inferior' : float
            Os valores recebidos em `p_superior`/`p_inferior` — iguais
            em todas as linhas, já que não variam por semente.
        'a1', 'b1', 'a2', 'b2' : float
            Declive e ordenada na origem de M(t) e de m(t), ajustadas
            aos pontos de todas as janelas combinadas.
    '''
    T = t[-1]
    sementes = []
    for w in w_list:
        t_res, x_res, t_sup, x_sup = [], [], [], []
        for inicio in range(0, len(x), w):
            jx, jt = x[inicio:inicio + w], t[inicio:inicio + w]
            if len(jx) < 2:
                continue
            lim_sup = np.percentile(jx, p_superior)
            lim_inf = np.percentile(jx, p_inferior)
            for j in np.where(jx >= lim_sup)[0]:
                t_res.append(jt[j]); x_res.append(jx[j])
            for j in np.where(jx <= lim_inf)[0]:
                t_sup.append(jt[j]); x_sup.append(jx[j])
        s = _ajustar_retas(t_res, x_res, t_sup, x_sup, 'percentil',
                           {'w': w, 'p_superior': p_superior, 'p_inferior': p_inferior}, T)
        if s:
            sementes.append(s)
    return pd.DataFrame(sementes)


def sementes_media_movel(t, x, w_list, n_std=1.5):
    '''Estratégia D — bandas de Bollinger.

    Com média móvel MM e desvio-padrão móvel DP (janela centrada de w
    observações), são candidatos a M(t) os pontos acima de MM + n_std * DP
    e a m(t) os pontos abaixo de MM - n_std * DP.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    w_list : sequência de int
        Comprimentos da janela centrada da média e do desvio-padrão móveis.
    n_std : float
        Número de desvios-padrão móveis que definem a largura da banda.

    Devolve
    -------
    DataFrame, com uma linha por semente admissível e as colunas:
        'metodo' : str
            Sempre 'media_movel'.
        'w' : int
            O comprimento de janela que gerou essa semente.
        'n_std' : float
            O valor recebido em `n_std` — igual em todas as linhas.
        'a1', 'b1', 'a2', 'b2' : float
            Declive e ordenada na origem de M(t) e de m(t) ajustadas.
        Tem, no máximo, len(w_list) linhas — uma tentativa por w.
        '''
    T = t[-1]
    sementes = []
    serie = pd.Series(x)
    for w in w_list:
        media = serie.rolling(window=w, center=True).mean()
        desvio = serie.rolling(window=w, center=True).std()
        idx_max = np.where(x > (media + n_std * desvio).values)[0]
        idx_min = np.where(x < (media - n_std * desvio).values)[0]
        s = _ajustar_retas(t[idx_max], x[idx_max], t[idx_min], x[idx_min],
                           'media_movel', {'w': w, 'n_std': n_std}, T)
        if s:
            sementes.append(s)
    return pd.DataFrame(sementes)


def gerar_todas_sementes(t, x, w_list, alpha_list, delta_list,
                         p_sup_list, p_inf_list, n_std_list):
    '''Aplica as quatro estratégias com todas as combinações de hiperparâmetros e
    junta as sementes num único DataFrame.

    As estratégias A e B (janela fixa e monótona) recebem as listas de
    hiperparâmetros diretamente e tratam a grelha de combinações
    internamente. As estratégias C e D (percentil e
    média móvel) só aceitam um percentil/desvio de cada vez, pelo que esta
    função as chama repetidamente: uma vez por combinação de
    p_sup_list × p_inf_list para C, uma vez por valor de n_std_list para D.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    w_list, alpha_list, delta_list : sequências
        Hiperparâmetros das estratégias A e B.
    p_sup_list, p_inf_list : sequências
        Percentis superior e inferior da estratégia C.
    n_std_list : sequência
        Números de desvios-padrão da estratégia D.

    Devolve
    -------
    DataFrame com todas as sementes admissíveis das quatro estratégias,
    unidas por concatenação. Como cada estratégia produz colunas de
    hiperparâmetros diferentes ('alpha'/'delta' para A e B, 'p_superior'/
    'p_inferior' para C, 'n_std' para D), as colunas que não se aplicam a
    uma dada linha ficam a NaN. O 'metodo' de cada linha indica que
    colunas de hiperparâmetros lhe são relevantes. As colunas 'metodo',
    'a1', 'b1', 'a2', 'b2' estão sempre preenchidas. DataFrame vazio se
    nenhuma estratégia produzir sementes admissíveis.
    '''
    tabelas = [sementes_janela_fixa(t, x, w_list, alpha_list, delta_list),
               sementes_janela_monotonica(t, x, w_list, alpha_list, delta_list)]
    for p_sup in p_sup_list:
        for p_inf in p_inf_list:
            tabelas.append(sementes_percentil(t, x, w_list, p_sup, p_inf))
    for n_std in n_std_list:
        tabelas.append(sementes_media_movel(t, x, w_list, n_std))
    nao_vazias = [tab for tab in tabelas if not tab.empty]
    return pd.concat(nao_vazias, ignore_index=True) if nao_vazias else pd.DataFrame()



def definir_espaco_procura(df_sementes, margem=0.5):
    '''Define o espaço de procura a partir das sementes, alargado por `margem`.

    Os declives ficam em [-A, A], com A = (1 + margem) * max|a|
    As ordenadas na origem ficam entre o menor e o maior b das sementes, alargados em
    margem * |b|. Os mesmos limites aplicam-se às duas retas.

    Parâmetros
    ----------
    df_sementes : DataFrame
        Sementes geradas por gerar_todas_sementes (colunas a1, b1, a2, b2).
    margem : float
        Fração de folga acrescentada além do intervalo coberto pelas
        sementes (assume-se >= 0; um valor negativo encolhe o espaço
        abaixo do intervalo das sementes).

    Devolve
    -------
    (LIMITE_INF, LIMITE_SUP, AMPLITUDE):
        LIMITE_INF, LIMITE_SUP : array de 4 elementos
            Limites inferior e superior do espaço de procura, na ordem
            [a1, b1, a2, b2].
        AMPLITUDE : array de 4 elementos
            LIMITE_SUP - LIMITE_INF, elemento a elemento.
    '''
    global LIMITE_INF, LIMITE_SUP, AMPLITUDE
    a_max = max(abs(df_sementes[['a1', 'a2']].min().min()),
                abs(df_sementes[['a1', 'a2']].max().max())) * (1 + margem)
    b_min = df_sementes[['b1', 'b2']].min().min()
    b_max = df_sementes[['b1', 'b2']].max().max()
    b_inf = b_min - abs(b_min) * margem
    b_sup = b_max + abs(b_max) * margem
    LIMITE_INF = np.array([-a_max, b_inf, -a_max, b_inf])
    LIMITE_SUP = np.array([a_max, b_sup, a_max, b_sup])
    AMPLITUDE = LIMITE_SUP - LIMITE_INF
    dentro = np.ones(len(df_sementes), dtype=bool)
    for j, p in enumerate(['a1', 'b1', 'a2', 'b2']):
        dentro &= df_sementes[p].between(LIMITE_INF[j], LIMITE_SUP[j]).values
    print(f"Sementes dentro do espaço de procura: {dentro.sum()}/{len(df_sementes)}")
    return LIMITE_INF, LIMITE_SUP, AMPLITUDE


def normalizar(theta):
    '''Converte theta da escala original para [0, 1]^4.

    Parâmetros
    ----------
    theta : array de 4 elementos
        Vetor (a1, b1, a2, b2) na escala original.

    Devolve
    -------
    Array de 4 elementos em [0, 1], relativo aos limites definidos por
    definir_espaco_procura.
    '''
    return (theta - LIMITE_INF) / AMPLITUDE


def desnormalizar(theta_norm):
    '''Converte theta_norm de [0, 1]^4 para a escala original.

    Parâmetros
    ----------
    theta_norm : array de 4 elementos
        Vetor (a1, b1, a2, b2) normalizado em [0, 1].

    Devolve
    -------
    Array de 4 elementos (a1, b1, a2, b2) na escala original.
    '''
    return theta_norm * AMPLITUDE + LIMITE_INF


def _largura_minima(theta, T):
    '''Menor valor de M(t) - m(t) em [0, T].

    Sendo M - m afim, o mínimo é atingido num dos extremos do intervalo.

    Parâmetros
    ----------
    theta : sequência (a1, b1, a2, b2)
        Declives e ordenadas na origem de M(t) e m(t).
    T : float
        Fim do intervalo considerado, em anos.

    Devolve
    -------
    float : largura mínima da banda em [0, T].
    '''
    a1, b1, a2, b2 = theta
    return min(b1 - b2, (a1 * T + b1) - (a2 * T + b2))


def _suporte_minimo(theta, T):
    '''Menor valor de m(t) em [0, T] (atingido num dos extremos, por m ser afim).

    Parâmetros
    ----------
    theta : sequência (a1, b1, a2, b2)
        Declives e ordenadas na origem de M(t) e m(t).
    T : float
        Fim do intervalo considerado, em anos.

    Devolve
    -------
    float : valor mínimo de m(t) em [0, T].
    '''
    _, _, a2, b2 = theta
    return min(b2, a2 * T + b2)


def simplex_regular(n, gamma, centro):
    '''Simplex regular de n + 1 vértices em R^n, com escala gamma, deslocado para.

    Parâmetros
    ----------
    n : int
        Dimensão do espaço (número de parâmetros).
    gamma : float
        Escala do simplex.
    centro : array de n elementos
        Ponto para o qual o simplex é deslocado.

    Devolve
    -------
    array (n + 1, n)
        Uma linha por vértice do simplex (n + 1 vértices), uma coluna por
        coordenada em R^n. Cada linha é um ponto candidato inicial,
        já escalado por `gamma` e deslocado para `centro` — a matriz
    '''
    Y = np.zeros((n + 1, n))
    soma_quadrados = 0
    for i in range(n):
        c = np.sqrt(1 - soma_quadrados)
        r = (-1/n - soma_quadrados) / c
        Y[i, i] = c
        for j in range(i + 1, n + 1):
            Y[j, i] = r
        soma_quadrados += r**2
    for i in range(n + 1):
        Y[i] = gamma * Y[i] + centro
    return Y


def _objetivo_normalizado(theta_norm, t, x, largura_min=0.0):
    '''Função a minimizar: -log L(theta), com theta_norm em escala normalizada.

    Configurações inadmissíveis recebem uma penalização da ordem de 1e12,
    crescente com a violação (para orientar o simplex de volta à região
    admissível):
      - theta_norm fora de [0, 1]^4;
      - suporte m(t) não positivo algures em [0, T];
      - largura da banda M(t) - m(t) inferior a largura_min algures em [0, T].

    Parâmetros
    ----------
    theta_norm : array de 4 elementos
        Candidato (a1, b1, a2, b2) normalizado em [0, 1]^4.
    t, x : array
        Instantes e preços da série.
    largura_min : float
        Largura mínima admissível da banda M(t) - m(t).

    Devolve
    -------
    float
        -log-verosimilhança do candidato, se admissível.
        Se inadmissível, uma penalização >= 1e12, tanto maior quanto mais
        grave a violação — sempre muito superior a qualquer
        -log-verosimilhança real, para nunca ser confundida com uma
        solução genuína pelo otimizador.
    '''
    PENALIZACAO = 1e12
    excesso = (np.sum(np.clip(-theta_norm, 0, None))
               + np.sum(np.clip(theta_norm - 1, 0, None)))
    if excesso > 0:
        return PENALIZACAO * (1.0 + excesso)
    theta = desnormalizar(theta_norm)
    T = t[-1]
    suporte_min = _suporte_minimo(theta, T)
    if suporte_min <= 0:
        return PENALIZACAO * (1.0 + abs(suporte_min))
    largura = _largura_minima(theta, T)
    if largura < largura_min:
        return PENALIZACAO * (1.0 + abs(largura - largura_min))
    _, _, log_lik = classificar_e_estimar(t, x, list(theta))
    return -log_lik


def estimar_limiares_afins(t, x, df_sementes, gamma=0.1, xatol=1e-4, fatol=1e-4,
                               maxiter=500, kappa_min=3):
    '''Nelder-Mead multi-start sobre as sementes.

    Cada semente dentro do espaço de procura é ponto de partida de um
    Nelder-Mead com simplex inicial regular de escala gamma (em [0, 1]^4). A
    largura mínima da banda é kappa_min movimentos diários típicos do preço
    (ver _variacao_diaria_tipica).

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série de treino.
    df_sementes : DataFrame
        Sementes geradas por gerar_todas_sementes.
    gamma : float
        Escala do simplex inicial do Nelder-Mead.
    xatol, fatol : float
        Tolerâncias de paragem do Nelder-Mead (dispersão dos vértices e da
        função objetivo, respetivamente).
    maxiter : int
        Número máximo de iterações por execução do Nelder-Mead.
    kappa_min : float
        Largura mínima da banda, em movimentos diários típicos do preço.

    Devolve
    -------
    (theta_final, log_lik, semente_vencedora):
        theta_final : array de 4 elementos, ou None
            O (a1, b1, a2, b2) de maior log-verosimilhança entre todas as
            sementes testadas, na escala original.
        log_lik : float ou None
            A log-verosimilhança de theta_final.
        semente_vencedora : dict, ou None
            A semente (linha de df_sementes) que deu origem a
            theta_final, com a chave adicional 'LogLik_final'.
    '''
    melhor_objetivo = np.inf
    theta_final, semente_vencedora = None, None
    largura_min = kappa_min * _variacao_diaria_tipica(x)
    for _, semente in df_sementes.iterrows():
        inicio = normalizar(np.array([semente['a1'], semente['b1'],
                                      semente['a2'], semente['b2']]))
        if np.any(inicio < 0) or np.any(inicio > 1):
            continue
        res = minimize(_objetivo_normalizado, inicio, args=(t, x, largura_min),
                       method='Nelder-Mead',
                       options={'initial_simplex': simplex_regular(4, gamma, inicio),
                                'xatol': xatol, 'fatol': fatol, 'maxiter': maxiter})
        if res.fun < melhor_objetivo:
            melhor_objetivo = res.fun
            theta_final = desnormalizar(res.x)
            semente_vencedora = semente.to_dict()
            semente_vencedora['LogLik_final'] = round(-res.fun, 4)
    log_lik = -melhor_objetivo if theta_final is not None else None
    return theta_final, log_lik, semente_vencedora

def _estimar_mbg(t, x):
    '''Estimação por máxima verosimilhança do MBG com um único regime.

    A log-verosimilhança é a da densidade lognormal de X.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.

    Devolve
    -------
    (log_lik, mu, sigma2):
        log_lik : float
            Log-verosimilhança maximizada.
        mu : float
            Estimativa anualizada da tendência.
        sigma2 : float
            Estimativa anualizada da variância.
    '''
    dt = t[1] - t[0]
    log_retornos = np.diff(np.log(x))
    n = len(log_retornos)
    sigma2 = np.var(log_retornos) / dt
    mu = np.mean(log_retornos) / dt + sigma2 / 2
    residuos = log_retornos - (mu - 0.5 * sigma2) * dt
    log_lik = (- np.sum(np.log(x[1:]))
               + n * (-0.5 * np.log(2 * np.pi * sigma2 * dt))
               - np.sum(residuos**2) / (2 * sigma2 * dt))
    return log_lik, mu, sigma2


def _classificar_regimes_fixos(x, M, m, regime_inicial=1):
    '''Classificação em regimes com limiares horizontais M e m. Equivale a
    _classificar_regimes com a1 = a2 = 0.

    Parâmetros
    ----------
    x : array
        Preços observados.
    M, m : float
        Limiares horizontais superior e inferior.
    regime_inicial : int
        Regime assumido no primeiro instante.

    Devolve
    -------
    regimes : array de inteiros em {1, 2}
        Um por observação de x, na mesma convenção de _classificar_regimes:
        regimes[k] é o regime vigente no início do intervalo que gerou o
        retorno terminado em x[k].
    '''
    n = len(x)
    regime = regime_inicial
    regimes = np.zeros(n, dtype=int)
    regimes[0] = regime
    for i in range(1, n):
        if regime == 1 and x[i-1] >= M:
            regime = 2
        elif regime == 2 and x[i-1] <= m:
            regime = 1
        regimes[i] = regime
    return regimes


def _variacao_diaria_tipica(x):
    '''Variação diária típica do preço, em unidades monetárias.

    Parâmetros
    ----------
    x : array
        Preços observados.

    Devolve
    -------
    float
        O desvio-padrão dos log-retornos vezes a mediana do preço
        Uma aproximação do movimento diário típico, em unidades monetárias.
        0.0 se x tiver 2 ou menos observações.
    '''
    log_retornos = np.diff(np.log(x))
    if len(log_retornos) < 2:
        return 0.0
    return float(np.std(log_retornos)) * float(np.median(x))


def _estimar_limiares_fixos(t, x, n_grid=100, min_frac_regime=0.0, kappa_min=3):
    '''Estimação do modelo de limiares constantes por pesquisa em grelha sobre (M, m).

    Implementação prática dos estimadores, em duas
    fases: (i) grelha com n_grid quantis da série entre os percentis 5 e 95;
    (ii) refinamento com n_grid x n_grid pontos em torno do melhor par.

    Um par (M, m) é admissível se:
      - m < M e M - m >= kappa_min * variação diária típica, o que exclui
        comutações que traduzem apenas ruído de curto prazo;
      - cada regime tem pelo menos duas observações e, pelo menos, uma fração
        min_frac_regime da amostra.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série de treino.
    n_grid : int
        Número de pontos da grelha, em cada uma das duas fases.
    min_frac_regime : float
        Fração mínima da amostra exigida em cada regime.
    kappa_min : float
        Largura mínima da banda, em movimentos diários típicos do preço.

    Devolve
    -------
    (theta_fixo, log_lik, M, m):
        theta_fixo : list [0.0, M, 0.0, m], ou None
            Os limiares constantes na mesma representação de theta
            usada no resto do código (a1 = a2 = 0, b1 = M, b2 = m).
            None se nenhum par (M, m) for admissível.
        log_lik : float
            A log-verosimilhança do par (M, m) devolvido. -inf se
            nenhum par for admissível.
        M, m : float, ou None
            Os limiares superior e inferior encontrados — os mesmos
            valores já embutidos em theta_fixo. None se nenhum par
            for admissível.
    '''
    dt = t[1] - t[0]
    n = len(x)
    largura_min = kappa_min * _variacao_diaria_tipica(x)

    def _log_lik_par(M, m):
        '''Log-verosimilhança do par de limiares constantes (M, m). Uso interno de
        _estimar_limiares_fixos: testa os dois regimes iniciais possíveis e
        devolve o maior valor entre classificações admissíveis, ou -inf.

        Parâmetros
        ----------
        M, m : float
            Limiares superior e inferior candidatos.

        Devolve
        -------
        float (melhor) : log-verosimilhança do melhor regime inicial, ou -inf
        se o par for inadmissível (m >= M, banda demasiado estreita, ou
        nenhum regime inicial produzir dois regimes com observações
        suficientes).
        '''
        if m >= M or (M - m) < largura_min:
            return -np.inf
        melhor = -np.inf
        for regime_inicial in [1, 2]:
            regimes = _classificar_regimes_fixos(x, M, m, regime_inicial)
            n1 = int(np.sum(regimes[1:] == 1))
            n2 = int(np.sum(regimes[1:] == 2))
            if n1 < 2 or n2 < 2:                           
                continue
            if n1 < min_frac_regime * n or n2 < min_frac_regime * n:
                continue
            _, lik = _mle_por_regime(x, regimes, dt)
            melhor = max(melhor, lik)
        return melhor

    # Fase 1: grelha de quantis
    x_ordenado = np.sort(x)
    grelha = np.unique(x_ordenado[np.linspace(int(0.05 * n), int(0.95 * n), n_grid, dtype=int)])
    if len(grelha) < 2:
        return None, -np.inf, None, None

    melhor_lik, M_hat, m_hat = -np.inf, None, None
    for i, M in enumerate(grelha):
        for m in grelha[:i]:                                  
            lik = _log_lik_par(M, m)
            if lik > melhor_lik:
                melhor_lik, M_hat, m_hat = lik, M, m
    if M_hat is None:
        return None, -np.inf, None, None

    # Fase 2: refinamento entre os vizinhos do ótimo na grelha
    def _grelha_fina(valor):
        '''Intervalo de refinamento em torno de `valor`, robusto a grelhas curtas.
        Uso interno de _estimar_limiares_fixos, na segunda fase da pesquisa.

        Parâmetros
        ----------
        valor : float
            Ponto em torno do qual se constrói a grelha refinada (tipicamente o
            melhor M ou m da primeira fase).

        Devolve
        -------
        array : grelha densa de n_grid pontos em torno de `valor`.
        '''
        i = int(np.searchsorted(grelha, valor))
        inf = grelha[max(0, i - 1)]
        sup = grelha[min(len(grelha) - 1, i + 1)]
        if sup <= inf:                                       
            passo = float(np.ptp(grelha)) / max(1, len(grelha) - 1)
            inf, sup = valor - passo, valor + passo
        return np.linspace(inf, sup, n_grid)

    for M in _grelha_fina(M_hat):
        for m in _grelha_fina(m_hat):
            if m >= M:
                continue
            lik = _log_lik_par(M, m)
            if lik > melhor_lik:
                melhor_lik, M_hat, m_hat = lik, M, m
    return [0.0, float(M_hat), 0.0, float(m_hat)], melhor_lik, M_hat, m_hat

def comparar_modelos(t, x, theta_afim, lik_afim, kappa_min=3):
    '''Compara por AIC e BIC o MBG (k = 2), os limiares constantes (k = 6) e os
    limiares afins (k = 8).

    O MBG e os limiares constantes são estimados aqui. O modelo afim não é
    reestimado: usam-se theta_afim e lik_afim já obtidos (tipicamente após
    selecionar_melhor_candidato), para que seja avaliado na sua máxima
    verosimilhança efetiva. Se não houver nenhum par (M, m) admissível, os
    limiares constantes são omitidos da tabela.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série de treino.
    theta_afim : sequência (a1, b1, a2, b2)
        Limiares afins já estimados.
    lik_afim : float
        Log-verosimilhança de theta_afim.
    kappa_min : float
        Largura mínima da banda, em movimentos diários típicos do preço
        (aplicada também à estimação dos limiares constantes).

    Devolve
    -------
    (tabela, theta_fixo):
        tabela : DataFrame
            Uma linha por modelo (MBG; Limiares Constantes, se admissível;
            Limiares Afins), com colunas k, Log-Lik, AIC, BIC, indicadores
            booleanos do melhor AIC/BIC, e Δ AIC / Δ BIC face ao melhor
            valor de cada critério.
        theta_fixo : list [0.0, M_hat, 0.0, m_hat], ou None
            Os limiares constantes estimados por _estimar_limiares_fixos.
            None exatamente quando nenhum par (M, m) foi admissível.
    '''
    n_obs = len(x) - 1

    lik_mbg, mu_mbg, sigma2_mbg = _estimar_mbg(t, x)
    print("  MBG:")
    print(f"    mu={mu_mbg:.6f}, sigma={np.sqrt(sigma2_mbg):.6f}, LogLik={lik_mbg:.4f}")

    print("  Limiares constantes (pesquisa em grelha):")
    theta_fixo, lik_fixo, M_hat, m_hat = _estimar_limiares_fixos(t, x, kappa_min=kappa_min)
    if M_hat is not None:
        print(f"    M={M_hat:.4f}, m={m_hat:.4f}, LogLik={lik_fixo:.4f}")
    else:
        print("    nenhum par (M, m) admissível: modelo omitido")

    a1, b1, a2, b2 = theta_afim
    print("  Limiares afins (estimativa fornecida):")
    print(f"    a1={a1:.4f}, b1={b1:.4f}, a2={a2:.4f}, b2={b2:.4f}, LogLik={lik_afim:.4f}")

    def aic(k, ll): 
        '''Critério de informação de Akaike. Uso interno de comparar_modelos.

        Parâmetros
        ----------
        k : int
            Número de parâmetros do modelo.
        ll : float
            Log-verosimilhança maximizada.

        Devolve
        -------
        float : AIC = 2k - 2*ll.
        '''
        return 2 * k - 2 * ll
    def bic(k, ll): 
        '''Critério de informação bayesiano. Uso interno de comparar_modelos.

        Parâmetros
        ----------
        k : int
            Número de parâmetros do modelo.
        ll : float
            Log-verosimilhança maximizada.

        Devolve
        -------
        float : BIC = k*log(n_obs) - 2*ll (n_obs capturado do âmbito envolvente).
        '''
        return k * np.log(n_obs) - 2 * ll

    modelos = [('MBG', 2, lik_mbg), ('Limiares Afins', 8, lik_afim)]
    if M_hat is not None and np.isfinite(lik_fixo):
        modelos.insert(1, ('Limiares Constantes', 6, lik_fixo))

    tabela = pd.DataFrame([{'Modelo': nome, 'k': k, 'Log-Lik': round(float(ll), 4),
                            'AIC': round(float(aic(k, ll)), 2),
                            'BIC': round(float(bic(k, ll)), 2)}
                           for nome, k, ll in modelos])
    tabela['Melhor AIC'] = tabela['AIC'] == tabela['AIC'].min()
    tabela['Melhor BIC'] = tabela['BIC'] == tabela['BIC'].min()
    tabela['Δ AIC'] = (tabela['AIC'] - tabela['AIC'].min()).round(2)
    tabela['Δ BIC'] = (tabela['BIC'] - tabela['BIC'].min()).round(2)
    return tabela, theta_fixo

def diagnosticar_regimes(t, x, theta, nivel=0.05, verbose=True):
    '''Estimativas, erros-padrão e intervalos de confiança de (mu_r, sigma_r),
    e testes à diferença entre regimes.

    Erros-padrão assintóticos, condicionais à classificação em regimes:
      SE(mu)    = sigma / sqrt(T_r);
      SE(sigma) = sigma / sqrt(2 n_r).
    O IC de mu usa a aproximação normal.
    O IC de sigma usa a distribuição qui-quadrado com n_r - 1 graus de liberdade.

    Testes entre regimes:
      - diferença de drift: teste z;
      - diferença de volatilidade: rácio sigma1/sigma2 e teste de Levene;
      - sobreposição dos IC de mu e de sigma;
      - rótulo qualitativo, atribuído pela primeira condição satisfeita:
          1. 'regime residual' — um regime tem menos de 5% das
             observações válidas ou menos de 20 observações;
          2. 'segregação de volatilidade' — sigma1/sigma2 > 2 ou < 0.5,
             e o teste de Levene rejeita igualdade de variâncias
             (p < 0.05);
          3. 'alternância de tendência' — o teste z rejeita igualdade
             de drift (p <= 0.05);
          4. 'regime indistinto' — nenhuma das condições anteriores.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    theta : sequência (a1, b1, a2, b2)
        Limiares usados para classificar os regimes.
    nivel : float
        Nível de significância dos testes e dos intervalos de confiança
        (1 - nível é o grau de confiança).
    verbose : bool
        Se True, imprime as tabelas e a classificação qualitativa.

    Devolve
    -------
    (tabela, testes):
        tabela : DataFrame
            Uma linha por regime (1 e 2). Para um regime com n_r
            observações válidas: N_obs, Tempo (anos), mu, IC de mu,
            SE_mu, sigma, IC de sigma, SE_sigma.
        testes : dict
            As comparações entre regimes (mu1 - mu2, z_mu, p_mu,
            sigma1/sigma2, p_Levene, sobreposição dos IC de mu e de
            sigma, e o rótulo qualitativo).
    '''
    resultados, regimes, log_lik = classificar_e_estimar(t, x, theta)
    dt = t[1] - t[0]
    log_retornos = np.diff(np.log(x))
    z_crit = stats.norm.ppf(1 - nivel / 2)

    linhas, retornos_regime = {}, {}
    for r in [1, 2]:
        mu, sigma, n_r = resultados[r]['mu'], resultados[r]['sigma'], resultados[r]['n_obs']
        if mu is None or sigma is None or n_r < 3:
            linhas[r] = {'Regime': r, 'N_obs': n_r, 'Aviso': 'observações insuficientes'}
            continue
        retornos_regime[r] = log_retornos[regimes[1:] == r]
        T_r = n_r * dt
        se_mu = sigma / np.sqrt(T_r)
        se_sigma = sigma / np.sqrt(2 * n_r)
        gl = n_r - 1
        linhas[r] = {
            'Regime': r, 'N_obs': n_r, 'Tempo (anos)': round(T_r, 2),
            'mu (anual)': round(mu, 4),
            'IC_mu_inf': round(mu - z_crit * se_mu, 4),
            'IC_mu_sup': round(mu + z_crit * se_mu, 4),
            'SE_mu': round(se_mu, 4),
            'sigma (anual)': round(sigma, 4),
            'IC_sigma_inf': round(sigma * np.sqrt(n_r / chi2.ppf(1 - nivel / 2, gl)), 4),
            'IC_sigma_sup': round(sigma * np.sqrt(n_r / chi2.ppf(nivel / 2, gl)), 4),
            'SE_sigma': round(se_sigma, 4),
        }
    tabela = pd.DataFrame([linhas[r] for r in [1, 2]])

    testes = {}
    if 1 in retornos_regime and 2 in retornos_regime:
        mu1, mu2 = resultados[1]['mu'], resultados[2]['mu']
        s1, s2 = resultados[1]['sigma'], resultados[2]['sigma']
        n1, n2 = resultados[1]['n_obs'], resultados[2]['n_obs']
        se_mu1, se_mu2 = s1 / np.sqrt(n1 * dt), s2 / np.sqrt(n2 * dt)
        se_s1, se_s2 = s1 / np.sqrt(2 * n1), s2 / np.sqrt(2 * n2)

        z_mu = (mu1 - mu2) / np.sqrt(se_mu1**2 + se_mu2**2)
        p_mu = 2 * (1 - stats.norm.cdf(abs(z_mu)))
        p_levene = stats.levene(retornos_regime[1], retornos_regime[2]).pvalue
        racio_sigma = s1 / s2

        def _sobrepoem(c1, se1, c2, se2):
            '''Verifica se dois intervalos de confiança normais se sobrepõem. Uso interno
            de diagnosticar_regimes.

            Parâmetros
            ----------
            c1, c2 : float
                Estimativas centrais dos dois regimes.
            se1, se2 : float
                Erros-padrão associados a c1 e c2.

            Devolve
            -------
            bool : True se os intervalos [c - z*se, c + z*se] dos dois regimes se
            sobrepuserem (z fixado no âmbito envolvente).
            '''
            inf1, sup1 = c1 - z_crit * se1, c1 + z_crit * se1
            inf2, sup2 = c2 - z_crit * se2, c2 + z_crit * se2
            return not (sup1 < inf2 or sup2 < inf1)

        sobrepoe_mu = _sobrepoem(mu1, se_mu1, mu2, se_mu2)
        sobrepoe_sigma = _sobrepoem(s1, se_s1, s2, se_s2)

        fracao_menor = min(n1, n2) / (n1 + n2)
        if fracao_menor < 0.05 or min(n1, n2) < 20:
            rotulo = 'regime residual'
        elif (racio_sigma > 2 or racio_sigma < 0.5) and p_levene < 0.05:
            rotulo = 'segregação de volatilidade'
        elif p_mu <= 0.05:
            rotulo = 'alternância de tendência'
        else:
            rotulo = 'regime indistinto'

        testes = {'mu1 - mu2': round(mu1 - mu2, 4), 'z_mu': round(z_mu, 3),
                  'p_mu': round(p_mu, 4), 'sigma1 / sigma2': round(racio_sigma, 3),
                  'p_Levene': round(p_levene, 4),
                  'IC_mu_sobrepoe': bool(sobrepoe_mu),
                  'IC_sigma_sobrepoe': bool(sobrepoe_sigma),
                  'rotulo': rotulo}

    if verbose:
        print(f"Log-verosimilhança da classificação: {log_lik:.2f}\n")
        print("--- Estimativas por regime ---")
        print(tabela.to_string(index=False))
        if testes:
            print("\n--- Diferença entre regimes ---")
            print(f"  Drift:        mu1-mu2 = {testes['mu1 - mu2']:+.4f} "
                  f"(z={testes['z_mu']}, p={testes['p_mu']})  IC sobrepõem-se: {testes['IC_mu_sobrepoe']}")
            print(f"  Volatilidade: s1/s2   = {testes['sigma1 / sigma2']} "
                  f"(Levene p={testes['p_Levene']})  IC sobrepõem-se: {testes['IC_sigma_sobrepoe']}")
            print(f"  Classificação: {testes['rotulo'].upper()}")
        print("\n  NOTA: valores-p condicionais à classificação estimada (otimistas). A evidência")
        print("  confirmatória de que os regimes acrescentam algo é a comparação AIC/BIC com o MBG.")
    return tabela, testes

_INDICE_PARAMETRO = {'a1': 0, 'b1': 1, 'a2': 2, 'b2': 3}


def mapa_identificabilidade(t, x, theta_otimo, parametros=('a1', 'b1'), n_grid=40,
                            semiamplitudes=None, delta_ll=2.0, c_amp=0.25, largura_min=0.0,
                            mostrar=True, guardar=None, titulo=None):
    '''Perfil bidimensional da log-verosimilhança em torno do ótimo.

    Os dois parâmetros indicados em `parametros` variam numa grelha
    n_grid x n_grid centrada no ótimo, enquanto os restantes ficam fixos. As
    semiamplitudes por omissão são proporcionais à amplitude A da série:
    c_amp * A para as ordenadas na origem e c_amp * A / T para os declives, de
    modo que ambas desloquem a fronteira da mesma quantidade em t = T. (Se a
    série for quase constante, A é substituída por um valor mínimo de
    salvaguarda.) Podem ser impostas por
    parâmetro através de `semiamplitudes`.

    Pontos inadmissíveis (m(t) não positivo ou banda mais estreita do que
    largura_min) ficam a NaN, tal como qualquer ponto em que a classificação
    e estimação em `classificar_e_estimar` falhe.
    O gráfico assinala o ótimo e o contorno
    LL = max(LL) - delta_ll.


    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    theta_otimo : sequência (a1, b1, a2, b2)
        Solução de referência, em torno da qual se constrói o mapa.
    parametros : tuple de 2 str
        Nomes dos dois parâmetros a variar, de entre 'a1', 'b1', 'a2', 'b2'.
    n_grid : int
        Número de pontos da grelha, em cada eixo.
    semiamplitudes : dict, opcional
        Semiamplitude a impor a cada parâmetro.
    delta_ll : float
        Definição do contorno de tolerância desenhado no gráfico.
    c_amp : float
        Fração da amplitude da série usada nas semiamplitudes.
    largura_min : float
        Largura mínima admissível da banda M(t) - m(t).
    mostrar : bool
        Se True, mostra o gráfico.
    guardar : str, opcional
        Caminho do ficheiro onde o gráfico é guardado.
    titulo : str, opcional
        Título do gráfico.


    Devolve
    -------
    (grelha1, grelha2, grelha_ll): matrizes n_grid x n_grid,
    com os valores testados do primeiro parâmetro (grelha1),
    do segundo (grelha2) e a log-verosimilhança correspondente
    (grelha_ll).
    '''
    theta_otimo = list(map(float, theta_otimo))
    T = float(t[-1])
    p1, p2 = parametros
    i1, i2 = _INDICE_PARAMETRO[p1], _INDICE_PARAMETRO[p2]
    v1, v2 = theta_otimo[i1], theta_otimo[i2]
    amplitude_serie = float(np.max(x) - np.min(x))
    if not np.isfinite(amplitude_serie) or amplitude_serie <= 0:
        amplitude_serie = max(1.0, abs(float(np.mean(x))) * 0.1)

    def _semiamplitude(nome):
        '''Semiamplitude da grelha para um parâmetro. Uso interno de
        mapa_identificabilidade: devolve o valor imposto em `semiamplitudes`, se
        indicado, ou o valor por omissão (c_amp * amplitude, ajustado por T para
        os declives).

        Parâmetros
        ----------
        nome : str
            Nome do parâmetro ('a1', 'b1', 'a2' ou 'b2').

        Devolve
        -------
        float : semiamplitude do parâmetro, na escala original.
        '''
        if semiamplitudes and nome in semiamplitudes:
            return float(semiamplitudes[nome])
        if nome in ('a1', 'a2'):
            return c_amp * amplitude_serie / max(T, 1e-9)
        return c_amp * amplitude_serie

    eixo1 = np.linspace(v1 - _semiamplitude(p1), v1 + _semiamplitude(p1), n_grid)
    eixo2 = np.linspace(v2 - _semiamplitude(p2), v2 + _semiamplitude(p2), n_grid)
    grelha1, grelha2 = np.meshgrid(eixo1, eixo2)
    grelha_ll = np.full(grelha1.shape, np.nan)
    for i in range(n_grid):
        for j in range(n_grid):
            theta = theta_otimo.copy()
            theta[i1], theta[i2] = grelha1[i, j], grelha2[i, j]
            if _suporte_minimo(theta, T) <= 0 or _largura_minima(theta, T) < largura_min:
                continue
            try:
                grelha_ll[i, j] = classificar_e_estimar(t, x, theta)[2]
            except Exception:
                pass

    if mostrar or guardar:
        ll_max = np.nanmax(grelha_ll)
        fig, ax = plt.subplots(figsize=(8, 6.5))
        cmap = plt.cm.viridis.copy()
        cmap.set_bad('lightgrey')
        malha = ax.pcolormesh(eixo1, eixo2, np.ma.masked_invalid(grelha_ll),
                              cmap=cmap, shading='auto')
        try:
            ax.contour(eixo1, eixo2, np.ma.masked_invalid(grelha_ll),
                       levels=[ll_max - delta_ll], colors='white', linewidths=1.2)
        except Exception:
            pass
        ax.plot(v1, v2, marker='*', color='red', markersize=16,
                markeredgecolor='white', label='ótimo')
        fig.colorbar(malha, ax=ax, label='log-verosimilhança')
        ax.set_xlabel(p1); ax.set_ylabel(p2)
        ax.set_title(titulo or f'Identificabilidade: {p1} vs {p2}')
        ax.legend(loc='upper right')
        plt.tight_layout()
        if guardar:
            plt.savefig(guardar, dpi=110, bbox_inches='tight')
        if mostrar:
            plt.show()
        plt.close(fig)
    return grelha1, grelha2, grelha_ll


def _refinar_localmente(t, x, theta_inicial, gamma=0.1, largura_min=0.0):
    '''Nelder-Mead local a partir de theta_inicial, com tolerâncias mais finas
    do que na estimação principal.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    theta_inicial : sequência (a1, b1, a2, b2)
        Ponto de partida do refinamento.
    gamma : float
        Escala do simplex inicial.
    largura_min : float
        Largura mínima admissível da banda M(t) - m(t).

    Devolve
    -------
    (theta, log_lik):
        theta : array (a1, b1, a2, b2)
            O ótimo local encontrado pelo Nelder-Mead, já desnormalizado
            para a escala original.
        log_lik : float
            A log-verosimilhança nesse ótimo.
        Ou (None, -inf), se theta_inicial estiver fora do espaço de
        procura, isto é, fora do hipercubo normalizado [0, 1]^4.
    '''
    inicio = normalizar(np.array(theta_inicial, float))
    if np.any(inicio < 0) or np.any(inicio > 1):
        return None, -np.inf
    T = t[-1]

    def _objetivo(theta_norm):
        '''Log-verosimilhança negada de theta_norm, para o Nelder-Mead local. Uso
        interno de _refinar_localmente: devolve um valor muito elevado (1e18) se
        theta_norm estiver fora do hipercubo normalizado ou a configuração for
        inadmissível.

        Parâmetros
        ----------
        theta_norm : array de 4 elementos
            Candidato (a1, b1, a2, b2) normalizado em [0, 1]^4.

        Devolve
        -------
        float : -log-verosimilhança (ou penalização, se inadmissível).
        '''
        if np.any(theta_norm < 0) or np.any(theta_norm > 1):
            return 1e18
        theta = desnormalizar(theta_norm)
        if _suporte_minimo(theta, T) <= 0 or _largura_minima(theta, T) < largura_min:
            return 1e18
        return -classificar_e_estimar(t, x, list(theta))[2]

    res = minimize(_objetivo, inicio, method='Nelder-Mead',
                   options={'initial_simplex': simplex_regular(4, gamma, inicio),
                            'xatol': 1e-5, 'fatol': 1e-5, 'maxiter': 800})
    return desnormalizar(res.x), -res.fun


def selecionar_melhor_candidato(t, x, candidatos, refinar=True, gamma=0.1,
                                largura_min=0.0, verbose=True):
    '''Devolve o candidato admissível de maior log-verosimilhança.

    Cada candidato é avaliado tal como está e, se refinar=True, também após
    refinamento local. Candidatos inadmissíveis são ignorados.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    candidatos : lista de sequências (a1, b1, a2, b2)
        Configurações candidatas a comparar.
    refinar : bool
        Se True, cada candidato é também avaliado após refinamento local
        (ver _refinar_localmente).
    gamma : float
        Escala do simplex inicial do refinamento.
    largura_min : float
        Largura mínima admissível da banda M(t) - m(t).
    verbose : bool
        Se True, imprime a log-verosimilhança de cada candidato e, no
        final, a solução escolhida.

    Devolve
    -------
    (melhor_theta, melhor_ll):
        melhor_theta : list [a1, b1, a2, b2], ou None
            O candidato de maior log-verosimilhança entre os admissíveis
            None se nenhum candidato for admissível.
        melhor_ll : float
            A log-verosimilhança de melhor_theta. -inf se nenhum
            candidato for admissível.
    '''
    T = t[-1]
    melhor_theta, melhor_ll = None, -np.inf
    for theta in candidatos:
        if _suporte_minimo(theta, T) <= 0 or _largura_minima(theta, T) < largura_min:
            continue
        ll = classificar_e_estimar(t, x, theta)[2]
        if ll > melhor_ll:
            melhor_ll, melhor_theta = ll, list(theta)
        if refinar:
            theta_ref, ll_ref = _refinar_localmente(t, x, theta, gamma, largura_min)
            if theta_ref is not None and ll_ref > melhor_ll:
                melhor_ll, melhor_theta = ll_ref, list(theta_ref)
        if verbose:
            print(f"  candidato {np.round(theta, 3)} -> LL={ll:.3f}")
    if verbose:
        print(f"\nMelhor solução: LL={melhor_ll:.3f}  theta={np.round(melhor_theta, 4)}")
    return melhor_theta, melhor_ll


def analisar_foco(grelha1, grelha2, grelha_ll, parametros, lik_otimo=None,
                  delta_ll=2.0, verbose=True):
    '''Caracteriza o máximo (foco) de um mapa de identificabilidade.

    A região de tolerância é a componente de {LL >= max(LL) - delta_ll} que
    contém o foco. A sua dimensão indica quão bem determinados estão os dois
    parâmetros. Se lik_otimo for indicado, compara-o com o foco (quando
    verbose=True), o que permite detetar que o otimizador não atingiu o
    máximo naquela direção.

    Parâmetros
    ----------
    grelha1, grelha2, grelha_ll : array
        Saída de mapa_identificabilidade para o mesmo par de parâmetros.
    parametros : tuple de 2 str
        Nomes dos dois parâmetros do mapa.
    lik_otimo : float, opcional
        Log-verosimilhança da solução de referência, para comparação com o foco.
    delta_ll : float
        Queda máxima de log-verosimilhança, face ao máximo do mapa, que
        define a região de tolerância.
    verbose : bool
        Se True, imprime o foco, a região de tolerância e a comparação com
        lik_otimo.

    Devolve
    -------
    foco : dict
        'LL' : float
            Log-verosimilhança do foco (máximo da grelha).
        <nome do 1.º parâmetro>, <nome do 2.º parâmetro> : float
            Coordenadas do foco (chaves com os nomes dados em `parametros`).
        'frac_tolerancia' : float
            Fração das células admissíveis da grelha ocupada pela
            região de tolerância. NaN se não houver células admissíveis.
        'extensao_<parâmetro>' : float
            Extensão (máximo - mínimo) da região de tolerância em cada
            parâmetro, na escala original.
    '''
    p1, p2 = parametros
    idx_max = np.unravel_index(np.nanargmax(grelha_ll), grelha_ll.shape)
    ll_max = float(grelha_ll[idx_max])
    foco = {'LL': ll_max, p1: float(grelha1[idx_max]), p2: float(grelha2[idx_max])}

    acima_limiar = np.isfinite(grelha_ll) & (grelha_ll >= ll_max - delta_ll)
    componentes, _ = label(acima_limiar)
    regiao = (componentes == componentes[idx_max])

    n_admissiveis = int(np.sum(np.isfinite(grelha_ll)))
    foco['frac_tolerancia'] = float(np.sum(regiao) / n_admissiveis) if n_admissiveis else np.nan
    foco[f'extensao_{p1}'] = float(np.ptp(grelha1[regiao]))
    foco[f'extensao_{p2}'] = float(np.ptp(grelha2[regiao]))

    if verbose:
        print(f"Foco (máximo do mapa): LL={foco['LL']:.3f}  "
              f"{p1}={foco[p1]:.3f}  {p2}={foco[p2]:.3f}")
        print(f"  região de tolerância (ΔLL ≤ {delta_ll:g}): "
              f"{foco['frac_tolerancia']:.1%} da grelha admissível")
        print(f"    extensão em {p1}: {foco[f'extensao_{p1}']:.3f}"
              f"   |  extensão em {p2}: {foco[f'extensao_{p2}']:.3f}")
        if lik_otimo is not None:
            print(f"\nÓtimo da estimação: LL={lik_otimo:.3f}  "
                  f"(diferença para o foco: {lik_otimo - foco['LL']:+.3f})")
            if lik_otimo >= foco['LL'] - 1e-6:
                print("  => o ótimo é pelo menos tão bom como o melhor ponto do mapa.")
            else:
                print("  => ATENÇÃO: o mapa tem um ponto melhor do que o ótimo.")
    return foco


def grafico_regimes(t, x, theta, mostrar=True, guardar=None, titulo=None):
    '''Série de preços colorida pelo regime inferido, com M(t) e m(t) sobrepostos.

    O segmento entre t[i] e t[i+1] recebe a cor de regimes[i+1], o regime sob
    o qual o respetivo incremento foi gerado (a mesma convenção de
    _mle_por_regime).

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da série.
    theta : sequência (a1, b1, a2, b2)
        Limiares usados para classificar os regimes.
    mostrar : bool
        Se True, mostra o gráfico.
    guardar : str, opcional
        Caminho do ficheiro onde o gráfico é guardado.
    titulo : str, opcional
        Título do gráfico.

    Devolve
    -------
    dict com:
        'n_transicoes' : int
            Número de mudanças de regime entre posições consecutivas de
            regimes.
        'frac_regime1', 'frac_regime2' : float
            Fração das observações classificadas em cada regime.
    '''
    _, regimes, _ = classificar_e_estimar(t, x, theta)
    a1, b1, a2, b2 = theta
    cores = {1: 'royalblue', 2: 'darkorange'}
    plt.figure(figsize=(12, 6))
    plt.plot(t, a1 * t + b1, 'r--', alpha=0.7, lw=1.5)
    plt.plot(t, a2 * t + b2, 'g--', alpha=0.7, lw=1.5)
    for i in range(len(t) - 1):
        plt.plot(t[i:i+2], x[i:i+2], color=cores[regimes[i+1]], lw=1.1)
    plt.xlabel('Tempo (anos)'); plt.ylabel('Preço')
    plt.title(titulo or 'Série colorida pelo regime inferido')
    plt.legend(handles=[
        Line2D([0], [0], color='royalblue', lw=2, label='Regime 1'),
        Line2D([0], [0], color='darkorange', lw=2, label='Regime 2'),
        Line2D([0], [0], color='red', lw=1.5, ls='--', label='M(t)'),
        Line2D([0], [0], color='green', lw=1.5, ls='--', label='m(t)')], loc='upper left', fontsize=12)
    plt.grid(True, alpha=0.3); plt.tight_layout()
    if guardar:
        plt.savefig(guardar, dpi=110, bbox_inches='tight')
    if mostrar:
        plt.show()
    plt.close()
    return {'n_transicoes': int(np.sum(regimes[1:] != regimes[:-1])),
            'frac_regime1': float(np.mean(regimes == 1)),
            'frac_regime2': float(np.mean(regimes == 2))}



## 2. Especificidades dos dados simulados

In [2]:
import io
from contextlib import redirect_stdout


def estimar_regime_conhecido(t, x, regimes):
    '''Estimador geométrico com classificação de regime conhecida.

    Identifica os tempos de transição a partir da sequência verdadeira de regimes,
    ajusta por OLS X_tau ~ tau ao limiar superior (transições 1->2) e ao inferior
    (2->1) e estima (mu, sigma) por regime.

    O ponto de travessia usado na regressão é a média, em ambas as coordenadas,
    entre a última observação ainda dentro da banda e a seguinte, já além da
    fronteira. No simulador a comutação em i é desencadeada por x[i-1] já ter
    ultrapassado a fronteira, pelo que o par que enquadra o cruzamento é
    (i-2, i-1).

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da trajetória simulada.
    regimes : array de inteiros em {1, 2}
        Sequência de regimes (conhecida por a série ser simulada).

    Devolve
    -------
    (theta, results, ll), ou None se houver menos de duas transições
    utilizáveis de algum dos dois tipos (superior ou inferior):
        theta : list [a1, b1, a2, b2]
            Declives e ordenadas na origem das duas retas (superior e
            inferior).
        results : dict
            Estimativas por regime, devolvidas por _mle_por_regime
            ('mu', 'sigma', 'n_obs' para cada regime 1 e 2).
        ll : float
            Log-verosimilhança da classificação verdadeira, com os
            parâmetros de regime estimados.
    '''
    dt = t[1] - t[0]
    idx = np.where(regimes[1:] != regimes[:-1])[0] + 1
    sup_t, sup_x, inf_t, inf_x = [], [], [], []
    for i in idx:
        if i < 2:
            continue
        t_cruz = 0.5 * (t[i-2] + t[i-1])
        x_cruz = 0.5 * (x[i-2] + x[i-1])
        if regimes[i-1] == 1 and regimes[i] == 2:
            sup_t.append(t_cruz); sup_x.append(x_cruz)
        elif regimes[i-1] == 2 and regimes[i] == 1:
            inf_t.append(t_cruz); inf_x.append(x_cruz)
    if len(sup_t) < 2 or len(inf_t) < 2:
        return None
    mM = LinearRegression().fit(np.array(sup_t).reshape(-1, 1), sup_x)
    mm = LinearRegression().fit(np.array(inf_t).reshape(-1, 1), inf_x)
    theta = [float(mM.coef_[0]), float(mM.intercept_),
             float(mm.coef_[0]), float(mm.intercept_)]
    results, ll = _mle_por_regime(x, regimes, dt)
    return theta, results, ll


def _resumo_vazio(nomes):
    '''Resumo de recuperação com zero réplicas válidas. Uso interno de
    resumo_recuperacao, para evitar IndexError quando não há estimativas.

    Parâmetros
    ----------
    nomes : sequência de str
        Nomes dos parâmetros a listar.

    Devolve
    -------
    DataFrame com uma linha por parâmetro e as mesmas colunas de
    resumo_recuperacao (parametro, verdadeiro, media_est, vies, desvio_padrao,
    rmse, n_val). Todas as colunas numéricas a NaN e n_val = 0.
    '''
    return pd.DataFrame([{'parametro': n, 'verdadeiro': np.nan, 'media_est': np.nan,
                          'vies': np.nan, 'desvio_padrao': np.nan, 'rmse': np.nan,
                          'n_val': 0} for n in nomes])

def resumo_recuperacao(estimativas, verdadeiros, nomes):
    '''Dado um array (M x k) de estimativas e o vetor verdadeiro (k,), calcula por
    parâmetro: média das estimativas, viés, desvio-padrão e RMSE. Se não houver
    estimativas válidas, devolve linhas com NaN.

    Valores não finitos são descartados coluna a coluna, pelo que o número de
    estimativas usadas (n_val) pode diferir entre parâmetros.

    Parâmetros
    ----------
    estimativas : array (M, k)
        Uma linha por réplica, uma coluna por parâmetro.
    verdadeiros : array (k,)
        Valores geradores de cada parâmetro.
    nomes : sequência de k str
        Nomes dos parâmetros, para rotular as linhas do resultado.

    Devolve
    -------
    DataFrame com uma linha por parâmetro e as colunas:
        parametro : nome do parâmetro.
        verdadeiro : valor gerador.
        media_est : média das estimativas finitas.
        vies : media_est - verdadeiro.
        desvio_padrao : desvio-padrão amostral das estimativas.
        rmse : raiz do erro quadrático médio face ao valor verdadeiro.
        n_val : número de estimativas finitas usadas nesse parâmetro.
    '''
    est = np.asarray(estimativas, float)
    if est.size == 0 or est.ndim != 2:
        return _resumo_vazio(nomes)
    linhas = []
    for j, nome in enumerate(nomes):
        col = est[:, j]; col = col[np.isfinite(col)]
        vt = verdadeiros[j]
        linhas.append({'parametro': nome, 'verdadeiro': round(vt, 4),
                       'media_est': round(float(np.mean(col)), 4),
                       'vies': round(float(np.mean(col) - vt), 4),
                       'desvio_padrao': round(float(np.std(col, ddof=1)), 4),
                       'rmse': round(float(np.sqrt(np.mean((col - vt)**2))), 4),
                       'n_val': len(col)})
    return pd.DataFrame(linhas)


def estudo_monte_carlo(theta0, phi0, T=2.0, dt=1/DIAS_UTEIS_ANO, M=200, metodo='conhecido',
                       w_list=(21, 63, 126), alpha_list=(0.1,), delta_list=(-5, 0, 5),
                       p_sup_list=(95, 99), p_inf_list=(5, 1), n_std_list=(1.5,),
                       gamma=0.1, x0=100.0, semente_base=0, verbose=True):
    '''Estudo Monte Carlo de recuperação de parâmetros com uma estratégia de
    estimação. Gera M trajetórias do processo com (theta0, phi0) conhecidos,
    estima em cada réplica e agrega viés, desvio-padrão, rmse e a cobertura
    empírica dos IC assintóticos a 95% de mu e sigma.

    Uma tentativa é descartada (e substituída por outra, com a semente
    seguinte) se: com metodo='conhecido', houver menos de duas transições de
    algum tipo; com metodo='desconhecido', não houver sementes ou o
    Nelder-Mead não devolver uma solução admissível; em ambos os casos, se
    algum regime ficar com menos de duas observações. Para ao fim 
    de 100*M + 500 tentativas, podendo então haver menos de M réplicas.

    Parâmetros
    ----------
    theta0 : sequência (a1, b1, a2, b2)
        Limiares verdadeiros (geradores).
    phi0 : tuple (mu1, mu2, sigma1, sigma2)
        Parâmetros de regime verdadeiros (geradores).
    T, dt : float
        Horizonte e passo temporal da simulação.
    M : int
        Número de réplicas válidas a obter.
    metodo : str
        'conhecido' (OLS + MLE sobre o regime verdadeiro) ou 'desconhecido'
        (Nelder-Mead multi-start, como em dados reais). Qualquer valor
        diferente de 'conhecido' é tratado como 'desconhecido'.
    w_list, alpha_list, delta_list, p_sup_list, p_inf_list, n_std_list :
        hiperparâmetros das estratégias de geração de sementes (só usados se
        metodo='desconhecido').
    gamma : float
        Escala do simplex inicial do Nelder-Mead.
    x0 : float
        Preço inicial de cada trajetória simulada.
    semente_base : int
        Semente da primeira tentativa, incrementada a cada tentativa.
    verbose : bool
        Se True, imprime os resumos de recuperação e de cobertura.

    Devolve
    -------
    dict com:
        'estimativas_theta' : array (n_ok, 4)
            Estimativas (a1, b1, a2, b2) de cada réplica válida.
        'estimativas_phi' : array (n_ok, 4)
            Estimativas (mu1, mu2, sigma1, sigma2) de cada réplica válida.
        'resumo_theta', 'resumo_phi' : DataFrame
            Saídas de resumo_recuperacao para theta e phi.
        'cobertura' : DataFrame
            Coluna 'cobertura_IC95' por parâmetro (mu1, mu2, sigma1, sigma2).
        'n_ok' : int
            Número de réplicas válidas obtidas.
        'M' : int
            Número de réplicas pedido.
    O número de tentativas só é conhecido pelo aviso impresso (verbose).
    '''
    mu1, mu2, s1, s2 = phi0
    p1, p2 = (mu1, s1), (mu2, s2)
    theta_true = list(theta0)
    phi_true = [mu1, mu2, s1, s2]
    ests_theta, ests_phi = [], []
    cob_mu, cob_sig = {1: 0, 2: 0}, {1: 0, 2: 0}
    n_ok = 0
    z = 1.96
    seed = semente_base
    tentativas = 0
    max_tentativas = 100 * M + 500
    while n_ok < M and tentativas < max_tentativas:
        tentativas += 1
        np.random.seed(seed); seed += 1
        t, x, reg = simular_processo(T, dt, x0, p1, p2, theta_true)
        if metodo == 'conhecido':
            r = estimar_regime_conhecido(t, x, reg)
            if r is None:
                continue
            th, results, _ = r
        else:
            with redirect_stdout(io.StringIO()):
                sem = gerar_todas_sementes(t, x, list(w_list), list(alpha_list),
                                           list(delta_list), list(p_sup_list),
                                           list(p_inf_list), list(n_std_list))
                if sem.empty:
                    continue
                definir_espaco_procura(sem)
                th, lik, _ = estimar_limiares_afins(t, x, sem, gamma=gamma)
            if th is None:
                continue
            results, _, _ = classificar_e_estimar(t, x, list(th))
        if results[1]['mu'] is None or results[2]['mu'] is None:
            continue
        n_ok += 1
        ests_theta.append(list(th))
        ests_phi.append([results[1]['mu'], results[2]['mu'],
                         results[1]['sigma'], results[2]['sigma']])
        for rr in [1, 2]:
            mu, sg, nr = results[rr]['mu'], results[rr]['sigma'], results[rr]['n_obs']
            se_mu, se_s = sg / np.sqrt(nr * dt), sg / np.sqrt(2 * nr)
            mu_t = mu1 if rr == 1 else mu2
            s_t = s1 if rr == 1 else s2
            if abs(mu - mu_t) <= z * se_mu:
                cob_mu[rr] += 1
            if abs(sg - s_t) <= z * se_s:
                cob_sig[rr] += 1

    res_theta = resumo_recuperacao(ests_theta, theta_true, ['a1', 'b1', 'a2', 'b2'])
    res_phi = resumo_recuperacao(ests_phi, phi_true, ['mu1', 'mu2', 'sigma1', 'sigma2'])
    cobertura = pd.DataFrame([
        {'parametro': 'mu1', 'cobertura_IC95': round(cob_mu[1] / n_ok, 3) if n_ok else np.nan},
        {'parametro': 'mu2', 'cobertura_IC95': round(cob_mu[2] / n_ok, 3) if n_ok else np.nan},
        {'parametro': 'sigma1', 'cobertura_IC95': round(cob_sig[1] / n_ok, 3) if n_ok else np.nan},
        {'parametro': 'sigma2', 'cobertura_IC95': round(cob_sig[2] / n_ok, 3) if n_ok else np.nan},
    ])
    if verbose:
        aviso = "" if n_ok == M else f"  (só {n_ok} em {tentativas} tentativas!)"
        print(f"Monte Carlo ({metodo}): {n_ok} séries válidas (M={M}, com reposição)"
              f" | T={T} anos, N≈{int(T/dt)} obs{aviso}")
        print("\n-- Recuperação dos limiares (theta) --")
        print(res_theta.to_string(index=False))
        print("\n-- Recuperação dos parâmetros por regime (phi) --")
        print(res_phi.to_string(index=False))
        print("\n-- Cobertura empírica dos IC assintóticos a 95% (alvo 0.95) --")
        print(cobertura.to_string(index=False))
    return {'estimativas_theta': np.array(ests_theta), 'estimativas_phi': np.array(ests_phi),
            'resumo_theta': res_theta, 'resumo_phi': res_phi, 'cobertura': cobertura,
            'n_ok': n_ok, 'M': M}


def _resumo_e_cobertura(ests_theta, ests_phi, cob_mu, cob_sig, n_ok, theta_true, phi_true):
    '''Constrói os DataFrames de recuperação e de cobertura a partir das listas
    acumuladas. Uso interno de estudo_monte_carlo_emparelhado (chamada uma vez
    por estratégia).

    Parâmetros
    ----------
    ests_theta : lista de listas [a1, b1, a2, b2]
        Estimativas de theta, uma por réplica válida.
    ests_phi : lista de listas [mu1, mu2, sigma1, sigma2]
        Estimativas de phi, uma por réplica válida.
    cob_mu, cob_sig : dict {1: int, 2: int}
        Contagens de réplicas em que o IC de mu (ou sigma) cobriu o valor
        verdadeiro, por regime.
    n_ok : int
        Número de réplicas válidas.
    theta_true, phi_true : sequência
        Valores geradores de theta (a1, b1, a2, b2) e de phi (mu1, mu2,
        sigma1, sigma2).

    Devolve
    -------
    (res_theta, res_phi, cobertura):
        res_theta, res_phi : DataFrame
            Resumos de recuperação de theta e de phi (ver resumo_recuperacao).
        cobertura : DataFrame
            Uma linha por parâmetro (mu1, mu2, sigma1, sigma2), com a coluna
            'cobertura_IC95' = contagem / n_ok. NaN se n_ok = 0.
    '''
    res_theta = resumo_recuperacao(ests_theta, theta_true, ['a1', 'b1', 'a2', 'b2'])
    res_phi = resumo_recuperacao(ests_phi, phi_true, ['mu1', 'mu2', 'sigma1', 'sigma2'])
    cobertura = pd.DataFrame([
        {'parametro': p, 'cobertura_IC95': round(c / n_ok, 3) if n_ok else np.nan}
        for p, c in [('mu1', cob_mu[1]), ('mu2', cob_mu[2]),
                     ('sigma1', cob_sig[1]), ('sigma2', cob_sig[2])]])
    return res_theta, res_phi, cobertura


def _acumular_cobertura(results, phi0, dt, cob_mu, cob_sig, z=1.96):
    '''Atualiza os contadores de cobertura para uma estimação. Uso interno de
    estudo_monte_carlo_emparelhado.

    Para cada regime, incrementa o contador de mu (ou de sigma) se o valor
    verdadeiro cair no intervalo.

    Parâmetros
    ----------
    results : dict
        Dicionário de estimativas por regime (o primeiro elemento devolvido
        por _mle_por_regime ou classificar_e_estimar): results[r] contém
        'mu', 'sigma' e 'n_obs', para r = 1, 2.
    phi0 : tuple (mu1, mu2, sigma1, sigma2)
        Parâmetros de regime verdadeiros (geradores).
    dt : float
        Passo temporal.
    cob_mu, cob_sig : dict {1: int, 2: int}
        Contadores a atualizar, por regime.
    z : float
        Quantil normal correspondente ao nível de confiança dos IC.

    Devolve
    -------
    None (atualiza cob_mu e cob_sig por referência).
    '''
    mu1, mu2, s1, s2 = phi0
    for rr in [1, 2]:
        mu, sg, nr = results[rr]['mu'], results[rr]['sigma'], results[rr]['n_obs']
        se_mu, se_s = sg / np.sqrt(nr * dt), sg / np.sqrt(2 * nr)
        mu_t = mu1 if rr == 1 else mu2
        s_t = s1 if rr == 1 else s2
        if abs(mu - mu_t) <= z * se_mu:
            cob_mu[rr] += 1
        if abs(sg - s_t) <= z * se_s:
            cob_sig[rr] += 1


PARES_IDENTIF = (('a1', 'b1'), ('a2', 'b2'), ('a1', 'a2'), ('b1', 'b2'))


def _identificabilidade_agregada(seeds, thetas_d, T, dt, x0, p1, p2, theta_true,
                                 pares=PARES_IDENTIF, n_grid=40):
    '''Mede a identificabilidade agregada sobre várias réplicas de Monte Carlo.

    Recalcula a série de cada réplica em `seeds` a partir da semente guardada
    e constrói, em torno do theta_desconhecido já estimado
    nessa réplica, o mapa de identificabilidade de cada
    par e a sua caracterização (analisar_foco).

    Agrega, por par de parâmetros, a média e o desvio-padrão amostral
    entre réplicas de frac_tolerancia e da extensão normalizada.
    A extensão normalizada é: extensao / (2 * semiamplitude), isto é, a fração
    da largura da grelha do mapa.

    Parâmetros
    ----------
    seeds : sequência de int
        Sementes das réplicas a reprocessar (mesmo comprimento de thetas_d;
        se diferirem, usa-se o menor).
    thetas_d : sequência de theta
        Estimativas (método desconhecido) já obtidas em cada réplica.
    T, dt : float
        Horizonte e passo temporal da simulação, em anos.
    x0 : float
        Preço inicial das trajetórias.
    p1, p2 : tuple (mu, sigma)
        Parâmetros de regime verdadeiros (geradores), regime 1 e regime 2.
    theta_true : sequência (a1, b1, a2, b2)
        Limiares verdadeiros (geradores).
    pares : sequência de tuplos de 2 str
        Pares de parâmetros a mapear (por omissão, PARES_IDENTIF:
        (a1, b1), (a2, b2), (a1, a2), (b1, b2)).
    n_grid : int
        Resolução de cada mapa de identificabilidade.

    Devolve
    -------
    agregado : dict
        Uma entrada por par, com chave "<p1>_<p2>" (por exemplo 'a1_b1'),
        cujo valor é um dict com:
            'n' : número de mapas bem-sucedidos.
            'frac_tolerancia_media', 'frac_tolerancia_dp' : média e
                desvio-padrão da fração da grelha admissível ocupada pela
                região de tolerância.
            'extensao_norm_media_<p>', 'extensao_norm_dp_<p>' : média e
                desvio-padrão da extensão normalizada, para cada um dos dois
                parâmetros <p> do par.
    '''
    C_AMP = 0.25
    dados_por_par = {f"{p[0]}_{p[1]}": {'frac_tolerancia': [], 'extensao_norm': {p[0]: [], p[1]: []}}
                     for p in pares}

    for seed, theta_d in zip(seeds, thetas_d):
        np.random.seed(seed)
        t, x, _ = simular_processo(T, dt, x0, p1, p2, theta_true)
        amplitude = float(np.max(x) - np.min(x))
        Tt = float(t[-1])
        semi = {'a1': C_AMP * amplitude / Tt, 'a2': C_AMP * amplitude / Tt,
                'b1': C_AMP * amplitude, 'b2': C_AMP * amplitude}
        for par in pares:
            chave = f"{par[0]}_{par[1]}"
            try:
                G1, G2, LL = mapa_identificabilidade(t, x, theta_d, parametros=par,
                                                     n_grid=n_grid, mostrar=False)
                foco = analisar_foco(G1, G2, LL, par, verbose=False)
            except Exception:
                continue
            dados_por_par[chave]['frac_tolerancia'].append(foco.get('frac_tolerancia', np.nan))
            for p in par:
                ext = foco.get(f'extensao_{p}')
                if ext is not None:
                    dados_por_par[chave]['extensao_norm'][p].append(ext / (2 * semi[p]))

    agregado = {}
    for chave, d in dados_por_par.items():
        ft = np.array(d['frac_tolerancia'], float)
        ft = ft[np.isfinite(ft)]
        entrada = {'n': len(ft),
                   'frac_tolerancia_media': float(np.mean(ft)) if len(ft) else np.nan,
                   'frac_tolerancia_dp': float(np.std(ft, ddof=1)) if len(ft) > 1 else np.nan}
        for p, vals in d['extensao_norm'].items():
            v = np.array(vals, float)
            v = v[np.isfinite(v)]
            entrada[f'extensao_norm_media_{p}'] = float(np.mean(v)) if len(v) else np.nan
            entrada[f'extensao_norm_dp_{p}'] = float(np.std(v, ddof=1)) if len(v) > 1 else np.nan
        agregado[chave] = entrada
    return agregado


def estudo_monte_carlo_emparelhado(theta0, phi0, T=2.0, dt=1/DIAS_UTEIS_ANO, M=100,
                                   w_list=(21, 63, 126), alpha_list=(0.1,),
                                   delta_list=(-5, 0, 5), p_sup_list=(95, 99),
                                   p_inf_list=(5, 1), n_std_list=(1.5,), gamma=0.1,
                                   x0=100.0, semente_base=0, max_tentativas=None,
                                   K_identif=20, n_grid_identif=40, verbose=True):
    '''Estudo Monte Carlo empraelhado: os métodos conhecido e desconhecido são
    avaliados exatamente nas mesmas M séries. Séries inválidas são
    substituídas por novas até se atingir M séries válidas, ou até se esgotar
    max_tentativas, caso em que se devolvem menos de M.

    Uma série só é aceite se for válida para amboa os métodos:
      - conhecido: pelo menos duas transições de cada tipo (superior e
        inferior) e pelo menos duas observações em cada regime(n_r >= 2);
      - desconhecido: pelo menos uma semente utilizável, solução admissível do
        Nelder-Mead e pelo menos duas observações em cada regime (n_r >= 2).

    Além disso, para as primeiras `K_identif` réplicas válidas, recalcula a
    identificabilidade em torno do theta_desconhecido estimado e agrega
    frac_tolerancia/extensão normalizada sobre essas réplicas (ver _identificabilidade_agregada).

    Parâmetros
    ----------
    theta0 : sequência (a1, b1, a2, b2)
        Limiares verdadeiros (geradores).
    phi0 : tuple (mu1, mu2, sigma1, sigma2)
        Parâmetros de regime verdadeiros (geradores).
    T, dt : float
        Horizonte e passo temporal da simulação, em anos.
    M : int
        Número de réplicas válidas (nos dois métodos) a obter.
    w_list, alpha_list, delta_list, p_sup_list, p_inf_list, n_std_list :
        hiperparâmetros das estratégias de geração de sementes.
    gamma : float
        Escala do simplex inicial do Nelder-Mead.
    x0 : float
        Preço inicial de cada trajetória simulada.
    semente_base : int
        Semente da primeira tentativa, incrementada a cada tentativa.
    max_tentativas : int, opcional
        Número máximo de tentativas. Por omissão, 100*M + 500.
    K_identif : int
        Número máximo de réplicas usadas na identificabilidade agregada (usam-se
        as primeiras K_identif réplicas válidas, ou todas, se forem menos).
    n_grid_identif : int
        Resolução de cada mapa de identificabilidade agregada.
    verbose : bool
        Se True, imprime os resumos de recuperação e de cobertura e, para a
        identificabilidade agregada, apenas a frac_tolerancia de cada par.

    Devolve
    -------
    dict com:
        'conhecido', 'desconhecido' : dict
            Um por método, cada um com 'estimativas_theta' e
            'estimativas_phi' (arrays n_ok x 4), 'resumo_theta' e
            'resumo_phi' (ver resumo_recuperacao), 'cobertura', 'n_ok' e 'M'.
            O de 'desconhecido' inclui ainda 'identificabilidade_agregada'
            (dict de _identificabilidade_agregada) e 'K_identif' (número
            efetivo de réplicas usadas.
        'M_obtido' : int
            Número de séries válidas obtidas.
        'tentativas' : int
            Número total de séries simuladas, válidas ou não.
        'sementes' : list de int
            Semente de cada série válida, pela ordem de aceitação.
    '''
    mu1, mu2, s1, s2 = phi0
    p1, p2 = (mu1, s1), (mu2, s2)
    theta_true, phi_true = list(theta0), [mu1, mu2, s1, s2]
    if max_tentativas is None:
        max_tentativas = 100 * M + 500

    est_th_c, est_ph_c, cob_mu_c, cob_sig_c = [], [], {1: 0, 2: 0}, {1: 0, 2: 0}
    est_th_d, est_ph_d, cob_mu_d, cob_sig_d = [], [], {1: 0, 2: 0}, {1: 0, 2: 0}
    sementes_usadas = []
    seed = semente_base
    tentativas = 0

    while len(sementes_usadas) < M and tentativas < max_tentativas:
        tentativas += 1
        np.random.seed(seed); seed += 1
        t, x, reg = simular_processo(T, dt, x0, p1, p2, theta_true)

        # validade e estimação pelo método conhecido
        r_con = estimar_regime_conhecido(t, x, reg)
        if r_con is None:
            continue
        th_c, res_c, _ = r_con
        if res_c[1]['mu'] is None or res_c[2]['mu'] is None:
            continue

        # validade e estimação pelo método desconhecido
        with redirect_stdout(io.StringIO()):
            sem = gerar_todas_sementes(t, x, list(w_list), list(alpha_list), list(delta_list),
                                       list(p_sup_list), list(p_inf_list), list(n_std_list))
            if sem.empty:
                continue
            definir_espaco_procura(sem)
            th_d, _, _ = estimar_limiares_afins(t, x, sem, gamma=gamma)
        if th_d is None:
            continue
        res_d, _, _ = classificar_e_estimar(t, x, list(th_d))
        if res_d[1]['mu'] is None or res_d[2]['mu'] is None:
            continue

        sementes_usadas.append(seed - 1)
        est_th_c.append(list(th_c))
        est_ph_c.append([res_c[1]['mu'], res_c[2]['mu'], res_c[1]['sigma'], res_c[2]['sigma']])
        _acumular_cobertura(res_c, phi0, dt, cob_mu_c, cob_sig_c)
        est_th_d.append(list(th_d))
        est_ph_d.append([res_d[1]['mu'], res_d[2]['mu'], res_d[1]['sigma'], res_d[2]['sigma']])
        _acumular_cobertura(res_d, phi0, dt, cob_mu_d, cob_sig_d)

    n_ok = len(sementes_usadas)
    rt_c, rp_c, cob_c = _resumo_e_cobertura(est_th_c, est_ph_c, cob_mu_c, cob_sig_c,
                                            n_ok, theta_true, phi_true)
    rt_d, rp_d, cob_d = _resumo_e_cobertura(est_th_d, est_ph_d, cob_mu_d, cob_sig_d,
                                            n_ok, theta_true, phi_true)
    conhecido = {'estimativas_theta': np.array(est_th_c), 'estimativas_phi': np.array(est_ph_c),
                 'resumo_theta': rt_c, 'resumo_phi': rp_c, 'cobertura': cob_c, 'n_ok': n_ok, 'M': M}
    desconhecido = {'estimativas_theta': np.array(est_th_d), 'estimativas_phi': np.array(est_ph_d),
                    'resumo_theta': rt_d, 'resumo_phi': rp_d, 'cobertura': cob_d, 'n_ok': n_ok, 'M': M}

    # identificabilidade agregada sobre K_identif réplicas
    k = min(K_identif, n_ok)
    if k > 0:
        with redirect_stdout(io.StringIO()):
            identif_agg = _identificabilidade_agregada(
                sementes_usadas[:k], est_th_d[:k], T, dt, x0, p1, p2, theta_true,
                n_grid=n_grid_identif)
    else:
        identif_agg = {}
    desconhecido['identificabilidade_agregada'] = identif_agg
    desconhecido['K_identif'] = k

    if verbose:
        aviso = "" if n_ok == M else f"  (só {n_ok} válidas em {tentativas} tentativas!)"
        print(f"Monte Carlo emparelhado: {n_ok} séries usadas nos dois métodos"
              f" | T={T} anos, N≈{int(T/dt)} obs{aviso}")
        print("\n[CONHECIDO] Recuperação dos limiares (theta):")
        print(rt_c.to_string(index=False))
        print("[CONHECIDO] Recuperação por regime (phi):")
        print(rp_c.to_string(index=False))
        print("[CONHECIDO] Cobertura IC 95%:", dict(zip(cob_c['parametro'], cob_c['cobertura_IC95'])))
        print("\n[DESCONHECIDO] Recuperação dos limiares (theta):")
        print(rt_d.to_string(index=False))
        print("[DESCONHECIDO] Recuperação por regime (phi):")
        print(rp_d.to_string(index=False))
        print("[DESCONHECIDO] Cobertura IC 95%:", dict(zip(cob_d['parametro'], cob_d['cobertura_IC95'])))
        print(f"\n[DESCONHECIDO] Identificabilidade agregada sobre {k} réplicas:")
        for chave, e in identif_agg.items():
            print(f"  {chave}: frac_tolerancia={e['frac_tolerancia_media']:.3f} "
                  f"(dp={e['frac_tolerancia_dp']:.3f}, n={e['n']})")
    return {'conhecido': conhecido, 'desconhecido': desconhecido,
            'M_obtido': n_ok, 'tentativas': tentativas, 'sementes': sementes_usadas}


def _cores_regime():
    '''Cores usadas nos gráficos para os dois regimes. Uso interno de
    grafico_serie_limiares.

    Devolve
    -------
    dict {1: 'royalblue', 2: 'darkorange'}.
    '''
    return {1: 'royalblue', 2: 'darkorange'}


def grafico_serie_limiares(t, x, regimes, theta, titulo, theta_verd=None,
                           mostrar=True, guardar=None):
    '''Desenha a série colorida pelo `regimes` fornecido, com os limiares `theta`
    (a cheio) e, opcionalmente, os limiares verdadeiros `theta_verd` (tracejado
    cinza) sobrepostos para comparação.

    Parâmetros
    ----------
    t, x : array
        Instantes e preços da trajetória.
    regimes : array de inteiros em {1, 2}
        Regime a colorir em cada instante.
    theta : sequência (a1, b1, a2, b2)
        Limiares desenhados a cheio.
    titulo : str
        Título do gráfico.
    theta_verd : sequência (a1, b1, a2, b2), opcional
        Limiares verdadeiros, desenhados a tracejado para comparação.
    mostrar : bool
        Se True, mostra o gráfico.
    guardar : str, opcional
        Caminho do ficheiro onde o gráfico é guardado.

    Devolve
    -------
    None. A figura é mostrada e/ou guardada conforme `mostrar` e `guardar`.
    '''
    cores = _cores_regime()
    a1, b1, a2, b2 = theta
    plt.figure(figsize=(12, 6))
    for i in range(len(t) - 1):
        plt.plot(t[i:i+2], x[i:i+2], color=cores[regimes[i+1]], lw=1.1, zorder=2)
    plt.plot(t, a1 * t + b1, color='red', lw=1.8, label='M(t) (usado)', zorder=3)
    plt.plot(t, a2 * t + b2, color='green', lw=1.8, label='m(t) (usado)', zorder=3)
    if theta_verd is not None:
        va1, vb1, va2, vb2 = theta_verd
        plt.plot(t, va1 * t + vb1, color='grey', ls='--', lw=1.4, zorder=1)
        plt.plot(t, va2 * t + vb2, color='grey', ls='--', lw=1.4, zorder=1)
    plt.xlabel('Tempo (anos)'); plt.ylabel('Preço')
    plt.title(titulo)
    handles = [Line2D([0], [0], color=cores[1], lw=2, label='Regime 1'),
               Line2D([0], [0], color=cores[2], lw=2, label='Regime 2'),
               Line2D([0], [0], color='red', lw=1.8, label='M(t)'),
               Line2D([0], [0], color='green', lw=1.8, label='m(t)')]
    if theta_verd is not None:
        handles.append(Line2D([0], [0], color='grey', ls='--', lw=1.4, label='limiares verdadeiros'))
    plt.legend(handles=handles, loc='upper left', fontsize=12)
    plt.grid(True, alpha=0.3); plt.tight_layout()
    if guardar:
        plt.savefig(guardar, dpi=110, bbox_inches='tight')
    if mostrar:
        plt.show()
    plt.close()

    
#  AVALIAÇÃO AUTOMÁTICA COMPLETA EM DADOS SIMULADOS
def avaliar_serie_simulada(theta0, phi0, T=2.0, dt=1/DIAS_UTEIS_ANO, x0=100.0,
                           M_mc=100,
                           fatias=PARES_IDENTIF,
                           n_grid_id=40,
                           semente_ref=42,
                           incluir_identificabilidade=True,
                           K_identif=20,
                           w_list=(21, 63, 126), alpha_list=(0.1,), delta_list=(-5, 0, 5),
                           p_sup_list=(95, 99), p_inf_list=(5, 1), n_std_list=(1.5,),
                           gamma=0.1, mostrar=True):
    '''Avaliação automática completa do modelo em dados simulados.

    Dados os parâmetros verdadeiros theta0 e phi0, executa e devolve, num
    único dicionário:
      Série criada — trajetória verdadeira colorida pelo regime verdadeiro.
      Recuperação numa réplica — métodos CONHECIDO e DESCONHECIDO.
      Identificabilidade — mapa da réplica de referência (ilustrativo).
      Monte Carlo emparelhado — viés, desvio-padrão, rmse, cobertura, e
          identificabilidade agregada sobre K_identif réplicas.


    Parâmetros
    ----------
    theta0 : sequência (a1, b1, a2, b2)
        Limiares verdadeiros (geradores).
    phi0 : tuple (mu1, mu2, sigma1, sigma2)
        Parâmetros de regime verdadeiros (geradores).
    T, dt : float
        Horizonte e passo temporal da simulação.
    x0 : float
        Preço inicial das trajetórias.
    M_mc : int
        Número de réplicas do estudo Monte Carlo emparelhado.
    fatias : sequência de tuplos de 2 str
        Pares de parâmetros para os mapas de identificabilidade.
    n_grid_id : int
        Resolução dos mapas de identificabilidade.
    semente_ref : int
        Semente da réplica de referência.
    incluir_identificabilidade : bool
        Se True, calcula os mapas de identificabilidade da réplica de
        referência. Não desliga a identificabilidade agregada de,
        para isso, usar K_identif = 0.
    K_identif : int
        Número máximo de réplicas usadas na identificabilidade agregada.
    w_list, alpha_list, delta_list, p_sup_list, p_inf_list, n_std_list :
        hiperparâmetros das estratégias de geração de sementes.
    gamma : float
        Escala do simplex inicial do Nelder-Mead.
    mostrar : bool
        Se True, imprime as tabelas e mostra os gráficos de cada passo.
        (analisar_foco imprime sempre o foco, mesmo com mostrar=False.)
        
    Devolve
    -------
    R : dict com:
        'theta0', 'phi0', 'T', 'N' : parâmetros da simulação.
        'replica' : dict da réplica de referência, com 't', 'x',
            'regimes_verdadeiros', 'n_transicoes', 'theta_conhecido' (None se
            as transições forem insuficientes) e, se houver estimação,
            'params_conhecido', 'loglik_conhecido', 'theta_desconhecido' (None
            se falhar), 'loglik_desconhecido' e 'params_desconhecido'.
        'identificabilidade' : dict {"<p1>_<p2>": {'foco': ...}} por par de
            fatias (vazio se omitida ou sem estimação desconhecida).
        'monte_carlo' : dict com 'conhecido' e 'desconhecido' (ver
            estudo_monte_carlo_emparelhado), 'M_obtido', 'tentativas',
            'sementes' e 'perda_eficiencia', um DataFrame com, por limiar,
            'verdadeiro', 'rmse_conhecido', 'rmse_desconhecido' e 'fator'
            (rmse_desconhecido / rmse_conhecido).
    '''
    mu1, mu2, s1, s2 = phi0
    theta0 = list(map(float, theta0))
    R = {'theta0': theta0, 'phi0': tuple(phi0), 'T': T, 'N': int(T/dt)}
    if mostrar:
        print("=" * 68)
        print("  AVALIAÇÃO EM DADOS SIMULADOS")
        print(f"  theta0 = {theta0}")
        print(f"  phi0   = (mu1={mu1}, mu2={mu2}, sigma1={s1}, sigma2={s2})")
        print(f"  T = {T} anos  (N ≈ {int(T/dt)} observações)")
        print("=" * 68)

    # Série criada
    np.random.seed(semente_ref)
    t, x, reg_true = simular_processo(T, dt, x0, (mu1, s1), (mu2, s2), theta0)
    n_trans = int(np.sum(reg_true[1:] != reg_true[:-1]))
    R['replica'] = {'t': t, 'x': x, 'regimes_verdadeiros': reg_true, 'n_transicoes': n_trans}
    if mostrar:
        print(f"\n[1] Série criada  (transições verdadeiras: {n_trans})")
        grafico_serie_limiares(t, x, reg_true, theta0,
                               titulo='Série criada — regimes e limiares verdadeiros')

    # Recuperação + gráficos de recriação
    if mostrar:
        print("Recuperação numa réplica (conhecido e desconhecido)")
    rep = R['replica']

    r_con = estimar_regime_conhecido(t, x, reg_true)
    if r_con is not None:
        th_c, res_c, ll_c = r_con
        rep['theta_conhecido'] = th_c
        rep['params_conhecido'] = res_c
        rep['loglik_conhecido'] = ll_c
    else:
        th_c = res_c = ll_c = None
        rep['theta_conhecido'] = None

    with redirect_stdout(io.StringIO()):
        sem = gerar_todas_sementes(t, x, list(w_list), list(alpha_list), list(delta_list),
                                   list(p_sup_list), list(p_inf_list), list(n_std_list))
        th_d = ll_d = None
        if not sem.empty:
            definir_espaco_procura(sem)
            th_d, ll_d, _ = estimar_limiares_afins(t, x, sem, gamma=gamma)
    rep['theta_desconhecido'] = th_d
    rep['loglik_desconhecido'] = ll_d

    res_d = None
    if th_d is not None:
        res_d, _, _ = classificar_e_estimar(t, x, list(th_d))
        rep['params_desconhecido'] = res_d

    if mostrar:
        print(f"    theta verdadeiro    : {np.round(theta0, 3)}")
        if th_c is not None:
            print(f"    theta (conhecido)   : {np.round(th_c, 3)}   LogLik={ll_c:.2f}")
            print(f"      mu1={res_c[1]['mu']:+.3f}({mu1}) mu2={res_c[2]['mu']:+.3f}({mu2}) "
                  f"s1={res_c[1]['sigma']:.3f}({s1}) s2={res_c[2]['sigma']:.3f}({s2})")
            reg_c = classificar_e_estimar(t, x, list(th_c))[1]
            grafico_serie_limiares(t, x, reg_c, th_c, theta_verd=theta0,
                                   titulo='Recriação pelo método CONHECIDO')
        else:
            print("    (regime conhecido: transições insuficientes nesta réplica)")
        if th_d is not None:
            print(f"    theta (desconhecido): {np.round(th_d, 3)}   LogLik={ll_d:.2f}")
            if res_d is not None and res_d[1]['mu'] is not None and res_d[2]['mu'] is not None:
                print(f"      mu1={res_d[1]['mu']:+.3f}({mu1}) mu2={res_d[2]['mu']:+.3f}({mu2}) "
                      f"s1={res_d[1]['sigma']:.3f}({s1}) s2={res_d[2]['sigma']:.3f}({s2})")
            reg_d = classificar_e_estimar(t, x, list(th_d))[1]
            grafico_serie_limiares(t, x, reg_d, th_d, theta_verd=theta0,
                                   titulo='Recriação pelo método DESCONHECIDO')
        else:
            print("    (método desconhecido: estimação falhou nesta réplica)")

    # Identificabilidade (réplica de referência, ilustrativa)
    R['identificabilidade'] = {}
    if incluir_identificabilidade and th_d is not None:
        if mostrar:
            print("Identificabilidade (mapas + focos sobre a réplica de referência)")
            print("    NOTA: apenas ilustrativo — a versão agregada sobre "
                  f"{K_identif} réplicas está em monte_carlo.desconhecido.identificabilidade_agregada")
        theta_ref, lik_ref = list(th_d), ll_d
        for par in fatias:
            G1, G2, LL = mapa_identificabilidade(t, x, theta_ref, parametros=par,
                                                 n_grid=n_grid_id, mostrar=mostrar)
            f1 = analisar_foco(G1, G2, LL, par, lik_otimo=lik_ref)
            R['identificabilidade'][f"{par[0]}_{par[1]}"] = {'foco': f1}
    elif incluir_identificabilidade and mostrar:
        print("Identificabilidade: omitida (sem estimação do método desconhecido)")

    # Monte Carlo emparelhado
    if mostrar:
        print("Estudo Monte Carlo emparelhado (mesmas séries nos dois métodos)")
    emp = estudo_monte_carlo_emparelhado(
        theta0, phi0, T=T, dt=dt, M=M_mc,
        w_list=w_list, alpha_list=alpha_list, delta_list=delta_list,
        p_sup_list=p_sup_list, p_inf_list=p_inf_list, n_std_list=n_std_list,
        gamma=gamma, x0=x0, K_identif=K_identif, n_grid_identif=n_grid_id, verbose=mostrar)
    mc_con, mc_des = emp['conhecido'], emp['desconhecido']
    R['monte_carlo'] = {'conhecido': mc_con, 'desconhecido': mc_des,
                        'M_obtido': emp['M_obtido'], 'tentativas': emp['tentativas'],
                        'sementes': emp['sementes']}
    comp = mc_con['resumo_theta'][['parametro', 'verdadeiro']].copy()
    comp['rmse_conhecido'] = mc_con['resumo_theta']['rmse'].values
    comp['rmse_desconhecido'] = mc_des['resumo_theta']['rmse'].values
    with np.errstate(divide='ignore', invalid='ignore'):
        comp['fator'] = (comp['rmse_desconhecido'] / comp['rmse_conhecido']).round(1)
    R['monte_carlo']['perda_eficiencia'] = comp
    if mostrar:
        print("\n  Perda de eficiência nos limiares (RMSE desconhecido / conhecido):")
        print(comp.to_string(index=False))

    if mostrar:
        print("\n" + "=" * 68 + "\n  FIM DA AVALIAÇÃO\n" + "=" * 68)
    return R

## Cenários

In [3]:
OPCOES = dict(M_mc=300, K_identif=20)
# M_mc: nº de réplicas do Monte Carlo emparelhado (mesmas para conhecido e desconhecido).
# K_identif: nº de réplicas usadas na identificabilidade agregada.
RES = {}

### A) Geometria

In [ ]:
RES['A1'] = avaliar_serie_simulada([0, 110, 0, 92], (0.4, -0.4, 0.18, 0.28), T=3.0, **OPCOES)   # limiares horizontais

In [ ]:
RES['A2'] = avaliar_serie_simulada([4, 110, 4, 92], (0.4, -0.4, 0.18, 0.28), T=3.0, **OPCOES)   # canal ascendente

In [ ]:
RES['A3'] = avaliar_serie_simulada([-4, 110, -4, 92], (0.4, -0.4, 0.18, 0.28), T=3.0, **OPCOES)   # canal descendente

In [ ]:
RES['A4'] = avaliar_serie_simulada([-1, 110, 1, 92], (0.4, -0.4, 0.18, 0.28), T=3.0, **OPCOES)   # banda a estreitar

In [ ]:
RES['A5'] = avaliar_serie_simulada([4, 102.5, 4, 97.5], (0.4, -0.4, 0.18, 0.28), T=3.0, **OPCOES)   # banda estreita

### B) Volatilidade

In [ ]:
RES['B1'] = avaliar_serie_simulada([4, 110, 4, 92], (0.4, -0.4, 0.02, 0.02), T=3.0, **OPCOES)   # quase nulo

In [ ]:
RES['B2'] = avaliar_serie_simulada([4, 110, 4, 92], (0.4, -0.4, 0.1, 0.12), T=3.0, **OPCOES)   # baixa

In [ ]:
RES['B3'] = avaliar_serie_simulada([4, 110, 4, 92], (0.4, -0.4, 0.4, 0.55), T=3.0, **OPCOES)   # elevada

### C) Separação de tendência

In [ ]:
RES['C1'] = avaliar_serie_simulada([4, 110, 4, 92], (0.5, -0.5, 0.18, 0.18), T=3.0, **OPCOES)   # regimes distintos

In [ ]:
RES['C2'] = avaliar_serie_simulada([4, 110, 4, 92], (0.15, -0.15, 0.18, 0.18), T=3.0, **OPCOES)   # separação ténue

### D) Casos de fronteira

In [ ]:
RES['D1'] = avaliar_serie_simulada([4, 110, 4, 92], (0.4, 0.15, 0.18, 0.28), T=3.0, **OPCOES)   # mu1, mu2 do mesmo sinal

### E) Duração da seríe

In [ ]:
RES['E1'] = avaliar_serie_simulada([4, 110, 4, 92], (0.4, -0.4, 0.18, 0.28), T=9.0, **OPCOES)   # T=9 anos (vs. A2 com T=3)

## Síntese entre cenários

In [ ]:
def tabela_sintese(RES):
    '''Recuperação dos limiares e dos parâmetros de regime, e cobertura empírica
    dos IC, por cenário e por método.

    Para os limiares (a1, b1, a2, b2) tabelam-se viés, desvio-padrão e RMSE; para
    os parâmetros de regime (mu1, mu2, sigma1, sigma2), viés, RMSE e cobertura
    empírica do IC a 95% (o desvio-padrão destes parâmetros é calculado em
    resumo_recuperacao, mas não entra nesta tabela). Os valores são os
    arredondados dos resumos de cada estudo.

    Parâmetros
    ----------
    RES : dict {nome_cenario: resultado de avaliar_serie_simulada}
        Resultados de todos os cenários da bateria.

    Devolve
    -------
    DataFrame com uma linha por cenário e por método (conhecido/desconhecido),
    pela ordem de RES, e as colunas:
        cenario, metodo, n_ok, M :
            identificação e número de réplicas válidas obtidas / pedidas.
        vies_<p>, dp_<p>, rmse_<p>, para p em a1, b1, a2, b2.
        vies_<p>, rmse_<p>, cobertura_<p>, para p em mu1, mu2, sigma1, sigma2.
    '''
    linhas = []
    for nome, R in RES.items():
        for metodo in ['conhecido', 'desconhecido']:
            mc = R['monte_carlo'][metodo]
            rt = mc['resumo_theta'].set_index('parametro')
            rp = mc['resumo_phi'].set_index('parametro')
            cob = mc['cobertura'].set_index('parametro')['cobertura_IC95']
            linha = {'cenario': nome, 'metodo': metodo, 'n_ok': int(mc['n_ok']), 'M': int(mc['M'])}
            for p in ['a1', 'b1', 'a2', 'b2']:
                linha[f'vies_{p}'] = rt.loc[p, 'vies']
                linha[f'dp_{p}'] = rt.loc[p, 'desvio_padrao']
                linha[f'rmse_{p}'] = rt.loc[p, 'rmse']
            for p in ['mu1', 'mu2', 'sigma1', 'sigma2']:
                linha[f'vies_{p}'] = rp.loc[p, 'vies']
                linha[f'rmse_{p}'] = rp.loc[p, 'rmse']
                linha[f'cobertura_{p}'] = cob.loc[p]
            linhas.append(linha)
    return pd.DataFrame(linhas)

In [ ]:
def tabela_eficiencia(RES):
    '''Perda de eficiência (rmse do método desconhecido sobre o rmse do método
    conhecido) dos limiares, por cenário.

    Parâmetros
    ----------
    RES : dict {nome_cenario: resultado de avaliar_serie_simulada}
        Resultados de todos os cenários da bateria.

    Devolve
    -------
    DataFrame com uma linha por cenário e as colunas
    'cenario', 'fator_a1', 'fator_b1', 'fator_a2' e 'fator_b2'.
    '''
    linhas = []
    for nome, R in RES.items():
        comp = R['monte_carlo']['perda_eficiencia'].set_index('parametro')['fator']
        linha = {'cenario': nome}
        for p in ['a1', 'b1', 'a2', 'b2']:
            linha[f'fator_{p}'] = comp.loc[p]
        linhas.append(linha)
    return pd.DataFrame(linhas)

## Identificabilidade agregada entre cenários

In [ ]:
def tabela_identificabilidade(RES):
    '''Identificabilidade agregada por par de parâmetros e por cenário: fração
    média das células admissíveis do mapa 2D que pertencem à região de
    tolerância (frac_tolerancia), agregada sobre as K réplicas.

    Parâmetros
    ----------
    RES : dict {nome_cenario: resultado de avaliar_serie_simulada}
        Resultados de todos os cenários da bateria.

    Devolve
    -------
    DataFrame com uma linha por cenário e por par de parâmetros, e as colunas
    'cenario', 'par' (por exemplo 'a1_b1'), 'K' (número efetivo de réplicas
    usadas), 'frac_tolerancia_media' e 'frac_tolerancia_dp'.
    '''
    linhas = []
    for nome, R in RES.items():
        agg = R['monte_carlo']['desconhecido']['identificabilidade_agregada']
        K = R['monte_carlo']['desconhecido']['K_identif']
        for par, info in agg.items():
            linhas.append({'cenario': nome, 'par': par, 'K': K,
                           'frac_tolerancia_media': info['frac_tolerancia_media'],
                           'frac_tolerancia_dp': info['frac_tolerancia_dp']})
    return pd.DataFrame(linhas)